# How many rides were picked up in each zone?

When I started this project, I called the target ride demand. I was counting completed pickups and treating that count as if it included everyone who wanted a ride. I hadn't accounted for someone who opens the app, sees a fare that's too high or a wait that's longer than expected, and decides to leave. If they never take the ride, I won't see them in the trip records. I also can't tell how many people made that choice from these records.

So I'm forecasting **completed pickups**. If 40 rides started in a zone between 8 and 9 a.m., that zone gets a count of 40 for the 8 a.m. hour. I'll start with one month to check how the files record pickup time and zone before I build several years of hourly counts.

I load API credentials from a local `.env` file. I tried Ticketmaster early on and used setlist.fm for the historical concert features.


### What do I need to work with these files?

I'll use DuckDB for the large trip queries and pandas for the smaller results, with geographic tools for connecting external data to TLC zones. Some helpers come from the project package, so this notebook belongs inside the repository rather than being a standalone file.


In [ ]:
import os
import time
import urllib.request
from pathlib import Path

import duckdb
import geopandas as gpd
import holidays
import numpy as np
import pandas as pd
import plotly.express as px
import requests
from dotenv import load_dotenv
from lightgbm import LGBMRegressor
from shapely.geometry import Point
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from ride_demand.features.geo import haversine_distance
from ride_demand.features.weather import (
    combine_weather_files,
    download_weather_with_retry,
)

### Checking the API setup

First I'll make sure the API variables were loaded. There is no need to print the keys themselves. A presence check catches a missing setting before the first request, though it does not establish that the credential is valid.


In [ ]:
load_dotenv()
TICKETMASTER_API_KEY = os.environ.get("TICKETMASTER_API_KEY")
SETLISTFM_API_KEY_1 = os.environ.get("SETLISTFM_1_API_KEY")

In [ ]:
print(SETLISTFM_API_KEY_1 is None)

In [ ]:
def json_schema(obj):
    if isinstance(obj, dict):
        return {key: json_schema(value) for key, value in obj.items()}

    elif isinstance(obj, list):
        if len(obj) > 0:
            return [json_schema(obj[0])]
        return []

    else:
        return type(obj).__name__

## What does one trip record tell me?

Before building the full history, I'll open one January 2023 file and look at its fields. I want to understand what a trip record contains and where problems might appear before writing transformations around it.


In [ ]:
# df = pd.read_parquet(
#     r"https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2023-01.parquet"
# )

### Finding the project files

I'll locate the project through `pyproject.toml` rather than type in my computer's path. Then I can build the event-data directory from that root and use the same structure when the project moves.


In [ ]:
current = Path.cwd()
tries = 0
while not (current / "pyproject.toml").exists() and tries < 10:
    current = current.parent
    tries += 1

assert (
    current / "pyproject.toml"
).exists(), "Could not find project root (pyproject.toml not found within 10 parent directories)"

PROJECT_ROOT = current
EVENTS_DIR = PROJECT_ROOT / "data" / "external" / "events"
EVENTS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
print(PROJECT_ROOT)

In [ ]:
# df.shape

In [ ]:
# df.dtypes

In [ ]:
# df.isnull().sum()

In [ ]:
# df["pickup_datetime"].min(), df["pickup_datetime"].max()

In [ ]:
# plt.hist(df.wav_match_flag)

In [ ]:
# df.trip_miles.max()

In [ ]:
# df.trip_time.max()

## Starting with one month of TLC trips

I'll test the downloader on one month first. It keeps the original Parquet file under `data/bronze` and skips a file already stored locally. Once that works, I can expand to the historical period without downloading everything again each time I reopen the notebook.

[NYC TLC trip record downloads and data dictionaries](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)


In [ ]:
def download_tlc_file(dataset: str, year: int, month: int) -> Path:
    """ "
    Running the function repeatedly should not
    keep downloading the same file when it already exists.
    """
    month_str = f"{month:02d}"
    filename = f"{dataset}_tripdata_{year}-{month_str}.parquet"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
    path_to_save = PROJECT_ROOT / "data" / "bronze" / filename

    # if the file does not already exist on my computer, download it.
    if not path_to_save.exists():
        path_to_save.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path_to_save)

    return path_to_save

Running it again should return the saved file instead of making the same download. I'll use that behavior throughout the collection, then validate the contents separately.


In [ ]:
path2 = download_tlc_file("fhvhv", 2021, 1)
print(path2)

### Can DuckDB read the full month?

Now I'll query the January 2021 file with DuckDB. A row count is a simple first check that the file can be read without loading millions of trips into pandas.


In [ ]:
con = duckdb.connect()

In [ ]:
con.execute(f"SELECT COUNT(*) FROM '{path2}'").fetchall()

DuckDB reads 11,908,468 trips from the file. That gives me a usable first month. Next I'll inspect the schema and the fields needed to assign those trips to hours and zones.


### Which columns do I need for the pickup count?

The pickup timestamp and location ID are the starting point for the target. I'll check their types alongside the other available fields before aggregating anything.


In [ ]:
data_info = con.execute(f"DESCRIBE SELECT * FROM '{path2}'").df()
data_info

There are 24 columns, with timestamps for the time fields and numeric types for locations, distances, and payments. The structure looks suitable. Values can still be invalid even when their types are correct, so I'll check missingness and ranges next.


### Are any pickups missing their time or zone?

Every counted trip needs a pickup time and zone. I'll check those two fields first, since a missing value there would make hourly assignment impossible.


In [ ]:
con.execute(f"""SELECT pickup_datetime 
                FROM '{path2}' 
                WHERE pickup_datetime IS NULL """).df().shape

In [ ]:
con.execute(f"""SELECT COUNT(*)
                FROM '{path2}'
                WHERE pickup_datetime IS NULL""").fetchall()

In [ ]:
con.execute(f"""SELECT COUNT(*) 
                FROM '{path2}' 
                WHERE PULocationID IS NULL""").fetchall()

Neither field is missing in this file. That removes one immediate concern for the January 2021 aggregation, although it does not replace coverage checks on the other months.


### Do the values look plausible?

I'll look at minima and maxima before choosing cleaning rules. This can show an unexpected date range or a value that deserves investigation without assuming all unusual values should be removed.


In [ ]:
def check_for_min_max(df, path) -> pd.DataFrame:
    results = []
    for i in range(df.shape[0]):
        col = df["column_name"][i]
        result = con.execute(f"SELECT MIN({col}), MAX({col}) FROM '{path}'").fetchall()
        outcome = {"column": col, "min_max": result}
        results.append(outcome)

    df_results = pd.DataFrame(results)

    return df_results

In [ ]:
check_for_min_max(data_info, path2)

The pickup dates stay within January 2021 as expected. Other ranges deserve more thought: distance reaches 738.95 miles, duration exceeds 16 hours, and some recorded payments are negative. I'll leave them alone for the pickup count for now, but revisit them before modeling costs or waiting time.


In [ ]:
con.execute(f"SELECT MIN(pickup_datetime), MAX(pickup_datetime) FROM '{path2}' ").fetchall()

## Expanding to the historical period

The single-month test works, so I'll collect the available files through June 2026. A failed request should not discard the months already downloaded. Later 2026 requests return HTTP 403 in this run; I'll record those failures rather than interpret them as months with no trips.


In [ ]:
paths = []
for year in range(2021, 2027):
    for month in range(1, 13):
        try:
            path = download_tlc_file("fhvhv", year, month)
            paths.append(path)
            print(f"Downloaded {path}")
        except Exception as e:
            print(f"Failed to download: {year} - {month:02d} - {e}")

In [ ]:
path3 = PROJECT_ROOT / 'data/bronze/fhvhv_tripdata_2025-12.parquet'
con.execute(f"SELECT MIN(trip_miles), MAX(trip_miles) FROM '{path3}'").fetchall()

In [ ]:
print(f"{PROJECT_ROOT}/data/bronze/test.parquet")

## How much did pickup volume change?

I'll begin with monthly volume for 2021. DuckDB can read the twelve files as one query, which gives me a small summary without first joining the raw trips in memory.


In [ ]:
con.execute(f"""
    SELECT date_trunc('month', pickup_datetime), COUNT(*)
    FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_2021-*.parquet'
    GROUP BY date_trunc('month', pickup_datetime)
    ORDER BY date_trunc('month', pickup_datetime)""").df()

Monthly trips rise from about 11.9 million in January to over 16 million by October. A model that treats the earliest level as fixed would miss that change. I'll extend the view before deciding how much is recurring seasonality and how much is a changing level.


### Putting the monthly counts in one table

Now I'll run the same monthly aggregation across every available file. This gives me one row per month to plot rather than a huge trip-level table.


In [ ]:
%%time
df_monthly = con.execute(f"""
    SELECT date_trunc('month', pickup_datetime) AS month, COUNT(*) AS trips
    FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
    GROUP BY date_trunc('month', pickup_datetime)
    ORDER BY date_trunc('month', pickup_datetime)
    """).df()

df_monthly.head()

In [ ]:
df_monthly.shape

The result covers 66 months, from January 2021 to June 2026. That matches the collected period. I'll use the plot to see what the summary numbers alone hide.


## Monthly Trip Volume Trend

I'll plot completed trips by month and look for persistent growth, recurring peaks, and unusual drops. These are served rides, so the chart cannot show all requests or rides people wanted but did not take.


In [ ]:
fig = px.line(
    df_monthly, x="month", y="trips", title="Monthly High-Volume FHV Trip Volume, 2021 - 2026"
)
# fig.add_vline(x="2021-03-17", line_dash="dash", line_color="gray")
# fig.add_annotation(
#     x="2021-03-01",
#     y=1.02,
#     yref="paper",
#     showarrow=True,
#     text="Stimulus deposits begin",
#     bgcolor="cyan",
# )

# fig.add_vline(x="2021-07-01", line_dash="dash", line_color="gray")
# fig.add_annotation(
#     x="2021-07-01",
#     y=0.8,
#     yref="paper",
#     showarrow=True,
#     text="Child Tax Credit begins",
#     bgcolor="lightgreen",
# )

# fig.add_vline(x="2021-09-04", line_dash="dash", line_color="gray")
# fig.add_annotation(
#     x="2021-09-04", y=0.95, yref="paper", showarrow=True, text="FPUC $300 ends", bgcolor="salmon"
# )
# fig.update_layout(width=1000, height=500)
fig.show()

The later years have more trips than early 2021, with noticeable changes from month to month. Reopening is plausible context for the early period, but the chart alone cannot separate that from other influences. For modeling, the practical point is that both time patterns and changing activity levels matter.


## What can the recorded fares tell me?

I'll also look at base fares relative to travel. Averaging each trip's fare per mile could give tiny trips too much influence, so I'll start with monthly totals of fare, miles, and time.


In [ ]:
df_passenger_fare_monthly = con.execute(f"""
    SELECT date_trunc('month', pickup_datetime) AS month,
           SUM(base_passenger_fare) AS total_fare, 
           SUM(trip_miles) AS total_miles,
           SUM(trip_time) AS total_seconds
    FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
    GROUP BY date_trunc('month', pickup_datetime)
    ORDER BY date_trunc('month', pickup_datetime)
    """).df()
pd.options.display.float_format = "{:,.2f}".format
df_passenger_fare_monthly

The monthly totals give me 66 observations to work with. I can now divide fare by the corresponding distance or duration instead of averaging thousands of unstable short-trip ratios.


### Calculate Fare per Mile and Fare per Minute

I'll calculate fare per mile and fare per minute from those totals. These are descriptions of the trips served in each month, not the rate a passenger would be quoted for an otherwise identical journey.


In [ ]:
df_passenger_fare_monthly["price_per_mile"] = (
    df_passenger_fare_monthly.total_fare / df_passenger_fare_monthly.total_miles
)

df_passenger_fare_monthly["total_minutes"] = df_passenger_fare_monthly.total_seconds / 60
df_passenger_fare_monthly["price_per_minute"] = (
    df_passenger_fare_monthly.total_fare / df_passenger_fare_monthly.total_minutes
)

df_passenger_fare_monthly

For January 2021, the ratios are about $3.81 per mile and $1.08 per minute. I'll compare them over time, keeping in mind that a change in trip mix can also change either ratio.


### Base Fare per Mile Trend

I'll start with fare per mile. The monthly curve should make it easier to see whether the increase is gradual or concentrated in a few periods.


In [ ]:
fig = px.line(
    df_passenger_fare_monthly,
    x="month",
    y="price_per_mile",
    # x='month', y = 'earning per mile',
    #   title=dict(text="Fare per Mile, 2021-2026",
    #              font="Arial",
    #              color="#054F5E"),
    template="plotly_white",
)
fig.update_traces(line=dict(width=2, color="#057b95"))
fig.update_layout(
    title=dict(
        text="Fare per Mile, 2021-2026", font=dict(size=20, family="Roman", color="#074958")
    ),
)
fig.update_xaxes(showgrid=False, title="Month")
fig.update_yaxes(showgrid=True, gridcolor="#e2e8f0", title="Price per Mile ($)", tickprefix="$")
fig.show()

The ratio rises to about $5.77 by June 2026. That is a substantial change in recorded base fare relative to miles, but it does not tell me how much the same trip's price changed. Distance, journey mix, and fare conditions may all contribute.


### Base Fare per Minute Trend

Next I'll view the same fares against total trip minutes. A different denominator may reveal a different pattern.


In [ ]:
fig = px.line(
    df_passenger_fare_monthly,
    x="month",
    y="price_per_minute",
    template="plotly_white",
)

fig.update_traces(line=dict(width=2, color="#057b95"))

fig.update_layout(
    title=dict(
        text="Fare per Minute, 2021-2026", font=dict(size=20, family="Roman", color="#074958")
    ),
)

fig.update_xaxes(showgrid=False, title="Month")

fig.update_yaxes(showgrid=True, gridcolor="#e2e8f0", title="Price per Minute ($)", tickprefix="$")

fig.show()

Fare per minute rises from about $1.08 to $1.41. I'll keep both summaries as context for later cost targets, rather than treat either as a fixed pricing schedule.


In [ ]:
# fig, ax = plt.subplots(figsize=(12, 6))
# sns.lineplot(data=df_passenger_fare_monthly, x="month", y="price_per_mile", ax=ax)
# ax.set_xlabel("Month")
# ax.set_ylabel("Fare per Mile")
# ax.set_title("Fare per Mile, 2021-2026")
# plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(12, 6))
# sns.lineplot(data=df_passenger_fare_monthly, x="month", y="price_per_minute", ax=ax)
# ax.set_xlabel("Month")
# ax.set_ylabel("Price per Minute")
# ax.set_title("Fare per Minute, 2021-2026")
# plt.show()

In [ ]:
# fig = px.line(
#     df_passenger_fare_monthly, x="month", y="price_per_mile", title="Fare per Mile, 2021-2026"
# )
# fig = style_chart(fig, y_title="Price per Mile")
# fig.show()

## Where should I ask for the weather?

To connect trips with weather and events, I need geography at the zone level. I'll load the TLC polygons and calculate a representative point for each zone. Centroids are calculated in a projected coordinate system first, then converted back to latitude and longitude for external requests.


In [ ]:
# 4 levels up: ingestion/ -> ride_demand/ -> src/ -> project root
# project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def download_zones_file() -> Path:
    # month_str = f"{month:02d}"
    filename = "tlc_zones.geojson"
    url = "https://data.cityofnewyork.us/api/geospatial/8meu-9t5y?method=export&format=GeoJSON"
    path_to_save = PROJECT_ROOT / "data" / "external" / filename

    if not path_to_save.exists():
        path_to_save.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path_to_save)

    return path_to_save

In [ ]:
zones_path = download_zones_file()

In [ ]:
zones_path.read_text()[:300]

In [ ]:
zones_gdf = gpd.read_file(zones_path)
zones_gdf.head()

In [ ]:
zones_gdf_projected = zones_gdf.to_crs(epsg=2263)
zones_gdf["centroid"] = zones_gdf_projected.centroid.to_crs(epsg=4326)

In [ ]:
print(zones_gdf.crs)
print(zones_gdf_projected.crs)

In [ ]:
zones_gdf["centroid"] = zones_gdf_projected.centroid.to_crs(epsg=4326)
zones_gdf["centroid"].head()

The centroid gives each zone a consistent query location. Before using the full set, I'll still need to check repeated zone IDs and zones missing from the geometry reference. One point is a practical summary of an area, not the location of every pickup.


For the first weather test, I use Central Park, LaGuardia, and JFK. Their coordinates are (40.7789, -73.9692), (40.7794, -73.8803), and (40.6413, -73.7781), respectively.


## My first try with three weather stations

My first idea is to assign each zone to its nearest of those three stations. It is simple, but some outer zones remain far from the chosen station. I'll inspect those assignments before deciding whether the simplicity is worth the geographic approximation.


In [ ]:
stations = {
    "Central Park": (40.7789, -73.9692),
    "LaGuardia": (40.7794, -73.8803),
    "JFK": (40.6413, -73.7781),
}

In [ ]:
len(zones_gdf)
zones_gdf["centroid"].iloc[0].x

In [ ]:
def closest_station(centroid, stations=stations):
    distances = {}
    for name, (lat, lon) in stations.items():
        distance = haversine_distance(centroid.y, centroid.x, lat, lon)
        distances[name] = distance

    return min(distances, key=distances.get)

In [ ]:
zones_gdf["nearest_station"] = zones_gdf["centroid"].apply(closest_station)

In [ ]:
zones_gdf[["zone", "borough", "nearest_station"]].head(10)

In [ ]:
zones_gdf[zones_gdf["borough"] == "Staten Island"][["borough", "zone", "nearest_station"]].head(10)

Staten Island ends up assigned to Central Park. That may be the nearest of these three choices, but it is still a weak representation of local conditions. I'll switch to coordinate-based historical weather queries rather than keep the three-station approach.


## What changes if each zone gets its own weather?

I'll request historical weather at the zone centroids from the [Open-Meteo archive endpoint](https://archive-api.open-meteo.com/v1/archive). This gives me retrospective gridded weather for describing the completed rides. Later forecasting needs a separate source of weather forecasts available before the target hour; this historical table cannot serve that purpose directly.


In [ ]:
# print(os.listdir(PROJECT_ROOT / "data" / "external" / "weather"))

In [ ]:
weather_paths = []

weather_dir = PROJECT_ROOT / "data" / "external" / "weather"

downloaded_ids = {int(f.split("_")[1]) for f in os.listdir(weather_dir) if f.startswith("zone_")}

missing = zones_gdf[~zones_gdf["locationid"].astype(int).isin(downloaded_ids)]

print(f"{len(missing)} zones still missing")

Every zone in the geometry list already has a local file. That saves another round of requests. It does not yet tell me whether the list includes every pickup zone or whether the saved time series are complete.


In [ ]:
print(
    haversine_distance(40.7789, -73.9692, 40.6413, -73.7781)
)  # Central Park to JFK, should be ~14-15 miles

### Test the Per-Zone Weather Downloader

I'll try the downloader on one zone before applying the same process more widely. Returning an existing file should let the conversion work continue without another API call.


In [ ]:
# weather (use a zone you already have downloaded, so it's instant via the idempotency check)
test_path = download_weather_with_retry(
    location_id=1,
    centroid=zones_gdf.iloc[0]["centroid"],
    start_date="2021-01-01",
    stop_date="2026-06-30",
)
print(test_path)

The test returns zone 1's file from the expected weather directory. I'll use that file to inspect the response structure next.


### Download Weather Data for Missing Zones

Only zones without saved files need a request. I'll use retries for rate limits and keep individual failures visible so one bad request does not stop the entire collection.


In [ ]:
missing = zones_gdf[~zones_gdf["locationid"].astype(int).isin(downloaded_ids)]  # recompute first
for _index, row in missing.iterrows():
    try:
        path = download_weather_with_retry(
            row["locationid"], row["centroid"], "2021-01-01", "2026-06-30"
        )
        weather_paths.append(path)
        print(f"Downloaded {path}")
    except Exception as e:
        print(f"FAILED zone {row['locationid']}: {e}")

This makes the collection easier to resume. The next task is to check what is inside the files, rather than assume a successful download means a complete and correctly keyed weather series.


In [ ]:
# first_row = zones_gdf.iloc[0]
# test_path = download_weather(
#     first_row["locationid"], first_row["centroid"], "2021-01-01", "2026-06-30"
# )
# print(test_path)

Now I can move from collecting files to building the hourly weather table.


## Combine the Per-Zone Weather Files

I'll inspect the `hourly` section of one response before combining the collection. That shows how its timestamps and variables should become DataFrame columns.


In [ ]:
import json

with open(
    PROJECT_ROOT / "data" / "external" / "weather" / "zone_1_weather_2021-01-01_to_2026-06-30.json"
) as f:
    data = json.load(f)

print(data.keys())
print(data["hourly"].keys())

In [ ]:
weather_df = pd.DataFrame(data["hourly"])
weather_df.head()

In [ ]:
weather_df.info()

In [ ]:
weather_df.time = pd.to_datetime(weather_df["time"])
weather_df.info()

In [ ]:
weather_df["location_id"] = 1

The example has 48,168 hourly rows. I'll convert time to datetime and add the zone ID so those values can be connected to pickups later.


### Combine the Per-Zone Weather Files

I'll apply the same conversion to the remaining files, then concatenate the results. The intended unit is one zone and hour, so that key will need checking before the join.


In [ ]:
all_weather = combine_weather_files(
    zones_gdf["locationid"].tolist(),
    "2021-01-01",
    "2026-06-30",
)
all_weather.shape

In [ ]:
all_weather.head()

In [ ]:
all_weather.to_parquet(PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet")

In [ ]:
test_load = pd.read_parquet(PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet")
test_load.shape

The combined table has about 12.7 million rows and seven variables. Saving and reloading the Parquet works. That is a useful storage check, but the later join will tell me whether the keys and coverage behave as expected.


## Keeping the hours with no pickups

Now I'll build the main outcome: completed pickups in each zone and hour. The quiet hours need to be represented too, not just hours with recorded trips.


#### My first count missed the quiet hours

The first grouping only returns hours where at least one pickup occurred. That is easy to overlook because every returned count is valid. The missing piece is the set of zone-hours with zero trips.


#### What the model would miss

A model cannot learn from a zero hour that has disappeared from its table. I'll add those combinations explicitly so the target includes quiet periods as well as busy ones.


#### Giving every zone and hour a row

I'll create every supported zone-hour combination, then join the observed counts onto that grid. Unmatched counts become zero where the trip source is complete. Missing source coverage is a different problem and should never be turned into zero activity by the same shortcut.


In [ ]:
zone_pickups_complete = con.execute(f"""
    WITH hours AS (
        SELECT unnest(generate_series(
            TIMESTAMP '2021-01-01 00:00:00',
            TIMESTAMP '2026-06-30 23:00:00',
            INTERVAL 1 HOUR
        )) AS hour
    ),
    zones AS (
        SELECT LocationID AS location_id
        FROM read_csv_auto('{PROJECT_ROOT}/data/external/taxi_zone_lookup.csv')
        WHERE LocationID NOT IN (264, 265)
    ),
    grid AS (
        SELECT h.hour, z.location_id
        FROM hours h
        CROSS JOIN zones z
    ),
    actual_pickups AS (
        SELECT date_trunc('hour', pickup_datetime) AS hour,
               PULocationID AS location_id,
               COUNT(*) AS total_pickups
        FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
        WHERE PULocationID NOT IN (264, 265)
        GROUP BY date_trunc('hour', pickup_datetime), PULocationID
    )
    SELECT
        grid.hour,
        grid.location_id,
        COALESCE(actual_pickups.total_pickups, 0) AS total_pickups
    FROM grid
    LEFT JOIN actual_pickups
        ON grid.hour = actual_pickups.hour
        AND grid.location_id = actual_pickups.location_id
    ORDER BY grid.hour, grid.location_id
    """).df()

In [ ]:
%%time
zone_pickups_hourly = con.execute(f"""
    SELECT date_trunc('hour', pickup_datetime) AS hour, 
            (PULocationID) AS location_id, 
            COUNT(*) AS total_pickups
    FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
    GROUP BY date_trunc('hour', pickup_datetime), (PULocationID)
    ORDER BY date_trunc('hour', pickup_datetime), (PULocationID)
    """).df()

In [ ]:
zone_pickups_complete.shape

In [ ]:
zone_pickups_hourly.shape

In [ ]:
old_sum_real_zones_only = zone_pickups_hourly[~zone_pickups_hourly["location_id"].isin([264, 265])][
    "total_pickups"
].sum()

new_sum = zone_pickups_complete["total_pickups"].sum()

print(old_sum_real_zones_only, new_sum, old_sum_real_zones_only == new_sum)

The completed grid contains 48,168 hours across 263 zones, giving 12,668,184 rows. The pickup total is unchanged at 1,229,274,774. So the extra rows represent previously omitted zero hours rather than extra trips.


In [ ]:
zone_pickups_hourly = zone_pickups_complete.copy()

In [ ]:
zone_pickups_hourly.head()

In [ ]:
print(zone_pickups_hourly["hour"].iloc[5])
print(zone_pickups_hourly["hour"].dtype)

In [ ]:
zone_pickups_hourly[zone_pickups_hourly["location_id"] == 3].head(10)

In [ ]:
zone_pickups_hourly.to_parquet(PROJECT_ROOT / "data" / "gold" / "zone_hour_pickups.parquet")

In [ ]:
test_load = pd.read_parquet(PROJECT_ROOT / "data" / "gold" / "zone_hour_pickups.parquet")
test_load.shape

## Temperature Unit Conversion

I'll keep Celsius and add Fahrenheit for interpretation in the NYC charts. The conversion below is straightforward, but keeping the source field makes the change easy to trace.

$$
F = C \times \frac{9}{5} + 32
$$


In [ ]:
all_weather["temperature_f"] = all_weather.temperature_2m * (9 / 5) + 32

In [ ]:
all_weather.to_parquet(PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet")

In [ ]:
all_weather[["temperature_2m", "temperature_f"]].head(3)

The sample conversion gives 39.02 F for 3.9 C, as expected. I'll now join weather and pickups, then check that the join has not changed the target's size.


## Why did the weather join add rows?

Weather should add columns, not pickup rows. I'll left-join it to the full grid on zone and hour, then compare row counts and missing values immediately.


In [ ]:
weather_joined = con.execute(f"""
    SELECT
        p.hour,
        p.location_id,
        p.total_pickups,
        w.temperature_f,
        w.precipitation,
        w.rain,
        w.snowfall,
        w.wind_speed_10m
    FROM '{PROJECT_ROOT}/data/gold/zone_hour_pickups.parquet' p
    LEFT JOIN '{PROJECT_ROOT}/data/silver/weather_hourly_all_zones.parquet' w
        ON p.location_id = w.location_id AND p.hour = w.time
""").df()

In [ ]:
print(weather_joined.shape)
print(weather_joined["temperature_f"].isnull().sum())

In [ ]:
weather_check = pd.read_parquet(
    PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet"
)
print(weather_check.duplicated(subset=["location_id", "time"]).sum())

The first join does change the size: it adds 144,504 rows and leaves missing temperatures too. So there is more than one issue to investigate. I'll start with repeated weather keys, then look for zones without a match.


In [ ]:
weather_joined[weather_joined["temperature_f"].isnull()]["hour"].describe()

#### Why did the join add rows?

I'll work backward through the merge rather than fill the missing values immediately. Are zone-hour keys unique? Do the ID types agree? Does the geometry cover the same zones as the pickup grid? Those checks should help separate the causes.


In [ ]:
dupes = weather_check[weather_check.duplicated(subset=["location_id", "time"], keep=False)]
print(dupes["location_id"].nunique())
print(dupes["location_id"].value_counts().head(10))

The duplicates are concentrated in zones 103 and 56. That is a useful clue: repeated geographic records are more plausible than random duplicate hours appearing only in those locations.

$$
(\text{location-id}, \text{time})
$$


In [ ]:
missing_weather_zones = set(zone_pickups_hourly["location_id"].unique()) - set(
    weather_check["location_id"].unique()
)

# print(missing_weather_zones)

In [ ]:
zones_gdf["locationid"].value_counts().head(10)

The geometry confirms it: zone 103 appears three times and zone 56 twice. Those repeated records can reproduce the same weather series. I'll make the weather key unique at the level the pickup join expects.


In [ ]:
print(zone_pickups_hourly["location_id"].dtype)
print(weather_check["location_id"].dtype)

In [ ]:
unique_zones = zones_gdf.drop_duplicates(subset="locationid")
print(len(unique_zones))  # should be 263, not 265+

In [ ]:
all_weather["location_id"] = all_weather["location_id"].astype(int)
all_weather.to_parquet(PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet")

There is also a type mismatch: one location ID column is integer and the other is object. I'll standardize them before comparing coverage, so the same ID is recognized consistently.


In [ ]:
all_weather = all_weather.drop_duplicates(subset=["location_id", "time"])
all_weather.to_parquet(PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet")

With the keys standardized, I'll remove repeated zone-hour weather rows. This should fix the row multiplication, though it will not create weather for zones that are genuinely absent.

$$
(\text{location-id}, \text{time})
$$


In [ ]:
weather_check2 = pd.read_parquet(
    PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet"
)
missing_weather_zones = set(zone_pickups_hourly["location_id"].astype(int).unique()) - set(
    weather_check2["location_id"].astype(int).unique()
)
print(missing_weather_zones)

Three missing zones remain: 57, 104, and 105. Now that the duplicate and type issues are separated out, I can address those geographic gaps directly.


In [ ]:
null_hours = weather_joined[weather_joined["temperature_f"].isnull()]["hour"]
print(null_hours.dt.hour.value_counts().head(10))

In [ ]:
print(zones_gdf[zones_gdf["locationid"].astype(int).isin([57, 105])])

In [ ]:
non_special_nulls = weather_joined[
    weather_joined["temperature_f"].isnull()
    & ~weather_joined["location_id"].isin([57, 105, 264, 265])
]
print(len(non_special_nulls))

In [ ]:
print(
    zone_pickups_hourly[zone_pickups_hourly["location_id"].isin([57, 105, 264, 265])][
        "total_pickups"
    ].sum()
)

pickup_zones_weather_missing = [
    57,
    104,
    105,
    264,
    265,
]  # was [57, 58, 105, 264, 265] -- 58 was a typo, real gap is 104
for zone in pickup_zones_weather_missing:
    total_pickups = zone_pickups_hourly[zone_pickups_hourly["location_id"] == zone][
        "total_pickups"
    ].sum()
    print(f"Total pickups in zone{zone}: {total_pickups}")

print(zone_pickups_hourly["total_pickups"].sum())

In [ ]:
weather_check2.head()

## Finding a weather proxy for zone 57

Zone 57 lacks a centroid in this reference. I'll use zone 56's hourly weather as a nearby proxy so the pickup zone stays in the dataset. I need to keep that choice documented, since it is a replacement rather than independent weather information.


In [ ]:
zone57_replacement = weather_check2[weather_check2["location_id"] == 56].copy()
zone57_replacement.head()

In [ ]:
zone57_replacement["location_id"] = 57
zone57_replacement.head()

In [ ]:
weather_check3 = pd.concat([weather_check2, zone57_replacement], ignore_index=True)

In [ ]:
weather_check3[weather_check3["location_id"] == 57].head()

In [ ]:
print(weather_check3[weather_check3["location_id"] == 57].shape)

The proxy gives zone 57 the expected 48,168 rows. The coverage problem is resolved under that assumption; the series still belongs to the donor location.


## Keeping both replacements for zones 104 and 105

I'll use zone 103 as the weather donor for zones 104 and 105. First I check the donor's temperature coverage, then give each copied series the recipient's ID.


In [ ]:
weather_check3[weather_check3["location_id"] == 103]["temperature_f"].isnull().sum()

In [ ]:
for i in [104, 105]:
    zone_replacement = weather_check3[weather_check3["location_id"] == 103].copy()
    zone_replacement["location_id"] = i
    weather_check3 = pd.concat([weather_check3, zone_replacement], ignore_index=True)

The earlier loop accidentally rebuilt from the original table each time, so the second replacement dropped the first. I'll append to the updated table instead. This is why checking both recipient zones after the loop matters.


In [ ]:
print(weather_check3[weather_check3["location_id"] == 104].shape)
print(weather_check3[weather_check3["location_id"] == 105].shape)

Both zones now retain all 48,168 hourly records. I'll carry these as documented proxies rather than describe them as separate downloads for each location.


In [ ]:
with open(
    PROJECT_ROOT / "data" / "external" / "weather" / "zone_1_weather_2021-01-01_to_2026-06-30.json"
) as f:
    sample = json.load(f)
print(sample["hourly_units"])

## Weather Unit Validation and Conversion

Before converting the remaining units, I'll read the response metadata. Rain and precipitation use millimeters, snowfall centimeters, and wind kilometers per hour. The source units decide the conversions, not the variable names alone.


In [ ]:
weather_check3["wind_speed_mph"] = weather_check3.wind_speed_10m * 0.621371
weather_check3["precipitation_in"] = weather_check3.precipitation * 0.0393701
weather_check3["rain_in"] = weather_check3.rain * 0.0393701
weather_check3["snowfall_in"] = weather_check3.snowfall * 0.393701

In [ ]:
weather_check3[
    [
        "wind_speed_10m",
        "wind_speed_mph",
        "precipitation",
        "precipitation_in",
        "rain",
        "rain_in",
        "snowfall",
        "snowfall_in",
    ]
].head(5)

In [ ]:
weather_check3[weather_check3["snowfall"] > 2].head()

In [ ]:
weather_check3[weather_check3["rain"] > 10].head()

## Trying the weather join again

I'll try the join again with unique weather keys, consistent IDs, and the three proxy series in place. The expected result is the same pickup grid with weather columns added.


In [ ]:
weather_check3.to_parquet(PROJECT_ROOT / "data" / "silver" / "weather_hourly_all_zones.parquet")

weather_joined = con.execute(f"""
    SELECT
        p.hour, 
        p.location_id, 
        p.total_pickups,
        w.temperature_f, 
        w.precipitation_in, 
        w.rain_in, 
        w.snowfall_in,
        w.wind_speed_mph
    FROM '{PROJECT_ROOT}/data/gold/zone_hour_pickups.parquet' p
    LEFT JOIN '{PROJECT_ROOT}/data/silver/weather_hourly_all_zones.parquet' w
        ON p.location_id = w.location_id AND p.hour = w.time
""").df()

print(weather_joined.shape)
print(weather_joined["temperature_f"].isnull().sum())

In [ ]:
zone_pickups_hourly.shape

In [ ]:
print("Start:", weather_joined["hour"].min())
print("End:", weather_joined["hour"].max())

In [ ]:
remaining_nulls = weather_joined[weather_joined["temperature_f"].isnull()]
print(remaining_nulls["location_id"].value_counts())

This time the row count stays at 12,668,184, and the temperature check finds no gaps. That resolves the problems seen in the first join. I'll keep checking the other fields as they enter later analyses.


In [ ]:
weather_joined.head()

## What can I learn from the calendar alone?

Hour, weekday, and month should capture some of the regular pickup cycle. I'll also flag weekends, weekday rush hours, and weekend nights. These are explicit calendar definitions that I can later test, rather than assumptions that every flagged hour is busy.


In [ ]:
weather_joined["hour_of_day"] = weather_joined.hour.dt.hour
weather_joined["day_of_week"] = weather_joined.hour.dt.day_of_week
weather_joined["is_weekend"] = weather_joined["day_of_week"].isin([5, 6])
weather_joined["month"] = weather_joined.hour.dt.month

In [ ]:
weather_joined[["hour", "day_of_week", "is_weekend", "hour_of_day"]].head(10)

In [ ]:
weekday_rush = ~weather_joined["is_weekend"]
morning_rush = weather_joined.hour_of_day.between(7, 9)
evening_rush = weather_joined.hour_of_day.between(16, 18)
weather_joined["is_weekday_rush_hour"] = weekday_rush & (morning_rush | evening_rush)

In [ ]:
weekend_night = weather_joined["is_weekend"]
late_hours = (weather_joined["hour_of_day"] >= 22) | (weather_joined["hour_of_day"] <= 2)
weather_joined["is_weekend_night"] = weekend_night & late_hours

In [ ]:
weather_joined.head()

In [ ]:
print(
    weather_joined[weather_joined["is_weekday_rush_hour"]]["day_of_week"].unique()
)  # should only show 0-4, never 5 or 6
print(
    weather_joined[weather_joined["is_weekend_night"]]["day_of_week"].unique()
)  # should only show 5 or 6

The weekday and weekend checks match the implemented rules. Weekend-night rows here mean Saturday and Sunday, which is worth stating because a Friday-night operating definition would be different.


## Holiday Features

Some dates do not behave like ordinary weekdays or weekends. I'll add New York's official and observed holidays, then separate indicators for selected special dates. Keeping the definitions separate should make later results easier to interpret.


In [ ]:
us_holidays = holidays.US(state="NY", years=range(2021, 2027))
print(list(us_holidays.items())[:5])

In [ ]:
weather_joined.hour.dt.date[:5]

In [ ]:
weather_joined["is_holiday"] = weather_joined["hour"].dt.date.apply(lambda day: day in us_holidays)

In [ ]:
unofficial_holidays_days = {
    (10, 31),  # halloween
    (12, 31),  # new year's eve
    (1, 1),  # new year's day
    (3, 17),  # st patrick's day
}

weather_joined["is_unofficial_holiday"] = weather_joined["hour"].dt.date.apply(
    lambda d: (d.month, d.day) in unofficial_holidays_days
)

In [ ]:
print(
    weather_joined[weather_joined["hour"].dt.date == pd.Timestamp("2022-07-04").date()][
        "is_holiday"
    ].unique()
)

July 4, 2022 is flagged correctly. That gives me a quick check of the date matching before moving to another calendar issue: the repeated daylight-saving hour.


## Daylight Saving Time Correction

When clocks move back, the local 1 AM label covers two real hours. A naive aggregation can combine their pickups and make that row look unusually high. This notebook uses neighboring weeks to adjust it.

That helps describe the historical series, but using a later week is a concern for forecasting. I need to revisit this with UTC-aware aggregation or a past-only rule before claiming that every derived lag could have been constructed at issuance.


In [ ]:
dst_dates = [
    "2021-11-07 01:00:00",
    "2022-11-06 01:00:00",
    "2023-11-05 01:00:00",
    "2024-11-03 01:00:00",
    "2025-11-02 01:00:00",
]

dst_date_list = [pd.Timestamp(d).date() for d in dst_dates]

is_dst_row = (weather_joined["hour"].dt.date.isin(dst_date_list)) & (
    weather_joined["hour_of_day"] == 1
)
is_dst_row.sum()

In [ ]:
seven_days = pd.Timedelta("7D")

after_lookup = weather_joined[["hour", "location_id", "total_pickups"]].copy()
after_lookup["hour"] = after_lookup["hour"] - seven_days
after_lookup = after_lookup.rename(columns={"total_pickups": "after_pickups"})

before_lookup = weather_joined[["hour", "location_id", "total_pickups"]].copy()
before_lookup["hour"] = before_lookup["hour"] + seven_days
before_lookup = before_lookup.rename(columns={"total_pickups": "before_pickups"})

weather_joined = weather_joined.merge(after_lookup, on=["hour", "location_id"], how="left")
weather_joined = weather_joined.merge(before_lookup, on=["hour", "location_id"], how="left")

In [ ]:
replacement = weather_joined.loc[is_dst_row, ["before_pickups", "after_pickups"]].mean(axis=1)
print("rows with no valid before/after neighbor:", replacement.isna().sum())

valid_replacement = replacement.dropna()
weather_joined.loc[valid_replacement.index, "total_pickups"] = valid_replacement.round().astype(int)

In [ ]:
for event in dst_dates:
    dst_event_pickups = weather_joined[
        (weather_joined["hour"] == event) & (weather_joined["location_id"] == 79)
    ]["total_pickups"]

    week_after_event_pickups = weather_joined[
        (weather_joined["hour"] == pd.Timestamp(event) + pd.Timedelta(days=7))
        & (weather_joined["location_id"] == 79)
    ]["total_pickups"]

    pickups_ratio = dst_event_pickups.iloc[0] / week_after_event_pickups.iloc[0]
    print(pickups_ratio)

In [ ]:
weather_joined = weather_joined.drop(columns=["after_pickups", "before_pickups"])

## Initial Concert Data Source — Ticketmaster

I'll first try Ticketmaster for concerts and other large events. Venue coordinates can be mapped to TLC zones, but the source also needs to cover the historical years in the trip data.


### Events (concerts) ticketmaster API

I'll test the historical event requests before building a larger downloader around them.


In [ ]:
venues = pd.DataFrame(
    {
        "venue_name": [
            "Madison Square Garden",
            "Barclays Center",
            "Yankee Stadium",
            "Citi Field",
            "USTA National Tennis Center",
        ],
        "latitude": [40.7503, 40.6827, 40.8296, 40.7570, 40.7500],
        "longitude": [-73.9933, -73.9752, -73.9262, -73.8458, -73.8475],
    }
)
venues

In [ ]:
zones_gdf.head()

In [ ]:
points = []
for i in range(len(venues)):
    point = Point(venues.iloc[i]["longitude"], venues.iloc[i]["latitude"])
    points.append(point)

# points

matching_location_ids = []
for p in range(len(points)):
    match = zones_gdf[zones_gdf.contains(points[p])]
    matching_location_ids.append(
        [
            venues["venue_name"].iloc[p],
            match["locationid"].iloc[0],
            match["zone"].iloc[0],
            match["borough"].iloc[0],
        ]
    )

venues_plus_location_ids = pd.DataFrame(
    matching_location_ids, columns=(["venue_name", "location_id", "zone", "borough"])
)
venues_plus_location_ids["location_id"] = venues_plus_location_ids["location_id"].astype(int)
venues_plus_location_ids

In [ ]:
def search_venue(keyword):
    response = requests.get(
        "https://app.ticketmaster.com/discovery/v2/venues.json",
        params={"keyword": keyword, "apikey": TICKETMASTER_API_KEY, "stateCode": "NY"},
    )
    return response.json()


venue_names = [
    "Madison Square Garden",
    "Barclays Center",
    "Yankee Stadium",
    "CitiField",
    "Billie Jean King National Tennis Center",
]

for name in venue_names:
    result = search_venue(name)
    venues = result.get("_embedded", {}).get("venues", [])
    print(f"--- {name} ---")
    for v in venues[:3]:  # show a few candidates, in case of ambiguous matches
        city = v.get("city", {}).get("name")
        print(f"  {v.get('name')} | id={v.get('id')} | {city}")
    print()

In [ ]:
def generate_quarters(start_year=2021, start_quarter=1, end_year=2026, end_quarter=2):
    quarters = []
    year, q = start_year, start_quarter
    while (year, q) <= (end_year, end_quarter):
        start_month = (q - 1) * 3 + 1
        end_month = start_month + 2
        start = f"{year}-{start_month:02d}-01T00:00:00Z"
        # last day of end_month, safely (handles Feb/30-day months)
        end_date = pd.Timestamp(year=year, month=end_month, day=1) + pd.offsets.MonthEnd(0)
        end = end_date.strftime("%Y-%m-%dT23:59:59Z")
        quarters.append((start, end))
        q += 1
        if q > 4:
            q = 1
            year += 1
    return quarters

In [ ]:
def fetch_ticketmaster_events(venue_name, venue_id, start_date, end_date, api_key):
    """
    Fetch all events for one venue within one date-range chunk.
    Saves raw JSON to disk; skips the API call entirely if that file
    already exists (same idempotent pattern as the weather downloader).
    """
    EVENTS_DIR = PROJECT_ROOT / "data" / "external" / "events"
    EVENTS_DIR.mkdir(exist_ok=True, parents=True)
    safe_name = venue_name.lower().replace(" ", "_")
    url = "https://app.ticketmaster.com/discovery/v2/events.json"
    filepath = EVENTS_DIR / f"{safe_name}_{start_date[:10]}_to_{end_date[:10]}.json"

    if not filepath.exists():
        result = requests.get(
            url,
            params={
                "apikey": api_key,
                "venueId": venue_id,
                "startDateTime": start_date,
                "endDateTime": end_date,
                "size": 200,
            },
        )
        data = result.json()

        if "errors" in data:
            print(f"ERROR fetching {venue_name} {start_date[:10]}: {data['errors'][0]['detail']}")
            return  # don't save — next run will retry this one

        total_pages = data.get("page", {}).get("totalPages", 1)
        if total_pages > 1:
            print(
                f"WARNING: {venue_name} {start_date[:10]} has {total_pages} pages — some events may be missing"
            )

        with open(filepath, "w") as f:
            json.dump(data, f)

        print(f"Fetched: {venue_name}, {start_date[:10]} to {end_date[:10]}")
    else:
        print(f"Already cached, skipping: {venue_name}, {start_date[:10]} to {end_date[:10]}")

In [ ]:
fetch_ticketmaster_events(
    "Madison square garden",
    "KovZpZA7AAEA",
    "2026-08-10T00:00:00Z",
    "2026-09-10T00:00:00Z",
    TICKETMASTER_API_KEY,
)

The tested requests return upcoming events but not the historical coverage I need. I'll look for another source rather than interpret an empty historical response as evidence that no events occurred.


## Can ESPN supply the historical games?

ESPN looks more promising for historical sports schedules. I'll start with the Knicks and see whether the response contains dates, venues, and starts that overlap the trip history. After that, I can decide how to approximate the period when spectators leave.


In [ ]:
response = requests.get(
    "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams/ny/schedule",
    params={"season": 2023},
)
data = response.json()
print(len(data.get("events", [])), "games found")
print(data.get("events", [{}])[0].get("date") if data.get("events") else "no events")

The response contains 82 games for the 2022–2023 season. Its label is 2023, even though the first game is in October 2022. I'll use actual event dates for the join rather than assume the season year equals the calendar year.


### Finding the Correct ESPN Team IDs

Before collecting more teams, I'll query their ESPN IDs. That is safer than guessing abbreviations and finding out later that the wrong schedule was downloaded.


In [ ]:
def list_teams(sport, league):
    response = requests.get(f"https://site.api.espn.com/apis/site/v2/sports/{sport}/{league}/teams")
    data = response.json()
    teams = data["sports"][0]["leagues"][0]["teams"]
    for t in teams:
        info = t["team"]
        print(f"{info['displayName']:30} id={info['id']:6} abbrev={info['abbreviation']}")

In [ ]:
# list_teams("basketball", "nba")   # find Knicks and Nets
# list_teams("hockey", "nhl")       # find Rangers and Islanders
# list_teams("baseball", "mlb")     # find Yankees and Mets

### Inspecting One Game Record

I'll inspect one full record first. The useful fields include start time, venue, home/away designation, and status. Attendance may be interesting too, but I need to distinguish historical facts from inputs that would have been known before a forecast.


In [ ]:
import json

# with open(PROJECT_ROOT / "data" / "external" / "events" / "new_york_knicks_2023_schedule.json") as f:
#     data = json.load(f)

one_event = data["events"][1]
print(list(one_event.keys()))
print()
print(json.dumps(one_event, indent=2)[:2000])  # first chunk, so it's readable

The home/away designation lets me keep games at the local venue. Attendance and final status describe what eventually happened, so their availability would need separate treatment in a live forecasting version.


### Checking Whether the Game Was Actually Played

A scheduled game may be postponed or canceled. I'll inspect status fields so the historical collection does not automatically treat every schedule entry as a played event. For future predictions, I would need the status as known at issuance rather than the final outcome.


In [ ]:
print(one_event["competitions"][0].get("status"))

The example is marked final and completed. That gives me a way to inspect played historical games, but it does not make completion a fact that was known before the event.


### Building a Reusable Schedule Downloader

Now I'll turn the successful request into a downloader for team, sport, league, and season. Caching the raw schedules means parsing changes will not require fetching every file again.


In [ ]:
def fetch_espn_schedule(team_name, sport, league, team_id, season):
    """
    Fetch one team's full schedule for one season. Saves raw JSON to
    disk; skips the API call if that file already exists — same
    idempotent pattern as download_weather_with_retry and
    fetch_ticketmaster_events.
    """
    EVENTS_DIR = PROJECT_ROOT / "data" / "external" / "events"
    EVENTS_DIR.mkdir(exist_ok=True, parents=True)
    safe_name = team_name.lower().replace(" ", "_")
    filepath = EVENTS_DIR / f"{safe_name}_{season}_schedule.json"
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/{sport}/{league}/teams/{team_id}/schedule"

    # TODO: if filepath already exists, print a skip message and return
    if not filepath.exists():
        result = requests.get(url, params={"season": season})
        data = result.json()

        events_key = data.get("events", [])
        if len(events_key) == 0:
            print(f"WARNING: {team_name} {season} has zero events")
            return

        with open(filepath, "w") as f:
            json.dump(data, f)

        print(f"Downloaded {team_name}'s {season} season")
    else:
        print(f"Already downloaded, skipping: {team_name}, season {season}")

### Choosing the NYC Teams

I'll begin with the Knicks, Nets, Rangers, Yankees, and Mets. Their home venues fit the NYC taxi-zone scope. The Giants and Jets play in New Jersey, and the Islanders need a more complicated historical venue mapping, so I'll leave them out of this version.


In [ ]:
teams = [
    ("New York Knicks", "basketball", "nba", "18"),
    ("Brooklyn Nets", "basketball", "nba", "17"),
    ("New York Rangers", "hockey", "nhl", "13"),
    ("New York Yankees", "baseball", "mlb", "10"),
    ("New York Mets", "baseball", "mlb", "21"),
]

for team_name, sport, league, team_id in teams:
    for season in range(2021, 2027):
        fetch_espn_schedule(team_name, sport, league, team_id, season)

The saved schedules are already there, so the downloader skips new requests. Five teams across six season labels should give 30 files. I'll check that simple expectation before parsing the games.

$$\
5 \times 6 = 30
$$\


In [ ]:
schedule_files = [
    f
    for f in (PROJECT_ROOT / "data" / "external" / "events").iterdir()
    if f.name.endswith("_schedule.json")
]
print(len(schedule_files))

All 30 files are present. That is the expected team-season coverage. It is time to inspect the records themselves, where date ranges and game status can still create problems.


### Approximating when spectators leave

The schedule gives me a start time, but pickups may be more affected when spectators leave. I'll approximate the end from the start and an assumed duration, then create a post-event interval. It will be a modeling assumption rather than an observed ending time.


In [ ]:
def estimate_game_duration(sport, game_date):
    if sport == "baseball":
        cutoff = pd.Timestamp("2023-01-01", tz="UTC")
        return (
            pd.Timedelta(hours=2, minutes=40)
            if game_date >= cutoff
            else pd.Timedelta(hours=3, minutes=10)
        )
    elif sport == "basketball":
        return pd.Timedelta(hours=2, minutes=15)
    elif sport == "hockey":
        return pd.Timedelta(hours=2, minutes=30)

I'll use 2 hours 15 minutes for basketball, 2 hours 30 minutes for hockey, and baseball assumptions of 3 hours 10 minutes before 2023 and 2 hours 40 minutes afterward. Later comparisons can tell me whether the resulting windows help prediction. They cannot verify when each game ended.


### Why I Did Not Include the New York Islanders

The Islanders are a different mapping problem because their home venue changes during the history. One fixed team-to-zone lookup would be misleading. I would need the actual venue by date before adding them consistently.


## Building the Historical Game Dataset

I'll connect each selected team to its venue's TLC zone and keep that lookup in one place. Teams sharing an arena can share a zone, which means I'll need a unique zone-hour key after expanding events.


In [ ]:
team_zone_map = {
    "New York Knicks": {"sport": "basketball", "location_id": 186, "team_id": "18"},
    "New York Rangers": {"sport": "hockey", "location_id": 186, "team_id": "13"},
    "Brooklyn Nets": {"sport": "basketball", "location_id": 181, "team_id": "17"},
    "New York Yankees": {"sport": "baseball", "location_id": 247, "team_id": "10"},
    "New York Mets": {"sport": "baseball", "location_id": 93, "team_id": "21"},
}

### Extracting the Fields I Need From Each Game

The parser will keep the fields needed for home games and skip away games. Event IDs let me check duplication, while scheduled starts give the basis for the approximate post-game period.


In [ ]:
def extract_game_data(event, team_id):
    competition = event["competitions"][0]
    competitors = competition["competitors"]

    home_competitor = None
    for c in competitors:
        if c["homeAway"] == "home":
            home_competitor = c

    if home_competitor["id"] != team_id:
        return None  # this team was away, not home — skip it

    event_id = event["id"]
    start_time = event["date"]
    season_year = event["season"]["year"]
    season_type = event["seasonType"]["name"]
    attendance = competition["attendance"]

    return {
        "event_id": event_id,
        "start_time": start_time,
        "season_year": season_year,
        "season_type": season_type,
        "attendance": attendance,
    }


test_example = extract_game_data(one_event, 21)
test_example

### Processing a Full Team Schedule

I'll apply that parser to a complete saved schedule and add the team's sport and venue zone. Once this works for one file, the remaining team-season files can follow the same route.


In [ ]:
def process_schedule_file(filepath, team_name):
    with open(filepath) as f:
        data = json.load(f)
        # data_df = pd.DataFrame(data['events'])

    team_info = team_zone_map[team_name]
    games = []

    for event in data["events"]:
        home_game = extract_game_data(event, team_info["team_id"])
        if home_game is not None:
            home_game["team_name"] = team_name
            home_game["team_id"] = team_info["team_id"]
            home_game["location_id"] = team_info["location_id"]
            home_game["sport"] = team_info["sport"]
            games.append(home_game)
    return games

### Combining All Teams and Seasons

Now I'll combine all parsed schedules and convert the start field to datetime. I need its time zone settled before grouping games by local date or assigning hourly flags.


In [ ]:
all_games = []

for team_name, _info in team_zone_map.items():
    for season in range(2021, 2027):
        safe_name = team_name.lower().replace(" ", "_")
        filepath = EVENTS_DIR / f"{safe_name}_{season}_schedule.json"
        all_games.extend(process_schedule_file(filepath, team_name))

games_df = pd.DataFrame(all_games)
games_df["start_time_et"] = pd.to_datetime(games_df["start_time"]).dt.tz_convert("America/New_York")
print(games_df.shape)
print(games_df["team_name"].value_counts())

The combined table has 1,739 home-game records. The baseball totals are above the rough 81-times-six expectation, which deserves investigation. I'll check possible explanations rather than treat the difference as either obviously correct or obviously duplicated.

$$\
81 \times 6 = 486
$$\


The expected totals are useful prompts, not validation by themselves. Season lengths, status, and rescheduling can complicate the comparison. I'll inspect those records before using the counts as evidence of real extra games.


## Why are the baseball counts higher than expected?

My first thought is that postseason games may be included. I'll check the season-type field before assuming that explains the baseball totals.


In [ ]:
games_df[games_df["team_name"].isin(["New York Yankees", "New York Mets"])][
    "season_type"
].value_counts()

It does not: all 1,018 Yankees and Mets records are labeled regular season. Next I'll look for repeated event IDs. If the same source game appears twice, that would be a more direct explanation.


In [ ]:
games_df[games_df["team_name"].isin(["New York Yankees", "New York Mets"])][
    "event_id"
].duplicated().sum()

There are no duplicated event IDs. That rules out one kind of repetition, but not postponed and rescheduled entries with different IDs. I'll break the counts down by season and inspect the dates.


In [ ]:
games_df[games_df["team_name"].isin(["New York Yankees", "New York Mets"])].groupby(
    ["team_name", "season_year"]
).size()

Several seasons exceed the rough expectation, so one unusual year is not the whole story. I'll look at dates with multiple entries next, while keeping status and rescheduling as unresolved possibilities.


In [ ]:
mets_2021 = games_df[(games_df["team_name"] == "New York Mets") & (games_df["season_year"] == 2021)]
mets_2021["start_time"].str[:10].duplicated().sum()

The first date check finds 11 same-day groups, but it uses UTC. A late New York game can fall on the next UTC date. I'll convert to local time before deciding which records share the same game day.


In [ ]:
mets_2021[mets_2021["start_time_et"].dt.date.duplicated(keep=False)].sort_values("start_time_et")[
    ["event_id", "start_time_et"]
]

The local-time check leaves nine distinct same-day pairs with separate IDs and starts. Those look like separate schedule entries rather than identical duplicates. They do not, by themselves, explain a season total above 81 home games: doubleheaders rearrange dates. Completion and rescheduling history still need review.

$$\
81 + 9 = 90
$$\


### When do pickups rise after a game?

I'll add the assumed duration to the New York start time and flag the following hour as the approximate departure window. New York local time changes between standard and daylight time, so I should not call the entire series EST.

$$\
\text{estimated end} = \text{game start time} + \text{estimated duration}
$$\

$$\
\text{window end} = \text{estimated end} + 1 \text{ hour}
$$\


In [ ]:
games_df["duration"] = games_df.apply(
    lambda row: estimate_game_duration(row["sport"], row["start_time_et"]), axis=1
)
games_df["estimated_end"] = games_df["start_time_et"] + games_df["duration"]
games_df["window_end"] = games_df["estimated_end"] + pd.Timedelta(hours=1)

In [ ]:
games_df.iloc[:, -4:].head()

An interval such as 9:45–10:45 PM touches two hourly pickup buckets. Assigning it only to 9 PM would miss most of the second hour. I'll mark every bucket with real overlap.


In [ ]:
hours = pd.date_range(start="2022-01-01 21:00", end="2022-01-02 01:00", freq="h")
print(hours)

### Converting the Post-Game Window Into Hourly Buckets

I'll use a helper to expand each continuous window into hourly timestamps. The endpoint rule matters here: a window ending exactly at 11 PM should not imply activity throughout the next hour.


In [ ]:
def get_affected_hours(estimated_end, window_end):
    """
    Given a game's estimated_end and window_end (both tz-aware),
    return every hour-bucket (as naive local timestamps, matching
    weather_joined) that overlaps the window at all.
    """
    # strip timezone info from both estimated_end and window_end
    estimated_end_strip_tz = estimated_end.tz_localize(None)
    window_end_strip_tz = window_end.tz_localize(None)

    # find the FIRST hour bucket to check — the hour containing
    # estimated_end, rounded DOWN to the top of that hour
    estimated_end_floor_hour = estimated_end_strip_tz.floor("h")
    window_end_floor_hour = window_end_strip_tz.floor("h")

    # build a list of candidate hours from that first bucket up
    window_hours = pd.date_range(
        start=estimated_end_floor_hour, end=window_end_floor_hour, freq="h"
    )
    filtered_hours = []
    for h in window_hours:
        if h < window_end_strip_tz and h + pd.Timedelta(hours=1) > estimated_end_strip_tz:
            filtered_hours.append(h)
    window_hours = filtered_hours

    return window_hours

### Building the Zone-Hour Sports Event Table

Expanding the windows gives me records at the level of the main table: venue zone and hour. A game can create more than one row, so later counts need to distinguish game records from flagged hours.


In [ ]:
demand_pairs = []
for _idx, row in games_df.iterrows():
    affected_hours = get_affected_hours(row["estimated_end"], row["window_end"])
    for h in affected_hours:
        demand_pairs.append({"location_id": row["location_id"], "hour": h})

demand_df = pd.DataFrame(demand_pairs)

In [ ]:
demand_df.shape

There are 3,443 expanded event-hour records before screening. I'll check their dates before joining them to a trip table that ends in June 2026.


### Checking the Sports Event Date Range

Sports seasons and the trip period do not have identical boundaries. Some downloaded games fall before January 2021 or after June 2026. I'll remove only those unmatched periods rather than describe them as invalid games.


In [ ]:
extra = demand_df[demand_df["hour"] > "2026-06-30 23:00:00"].copy()
extra.shape

In [ ]:
extra.head()

In [ ]:
demand_df[demand_df["hour"] < "2021-01-01"].shape[0]

The check finds 10 event-hour records before the period and 164 afterward. Removing them makes the coverage line up with the pickup table.


In [ ]:
demand_df = demand_df[
    (demand_df["hour"] >= "2021-01-01") & (demand_df["hour"] <= "2026-06-30 23:00:00")
]

### Checking for Duplicate Zone-Hour Event Records

Before merging, I'll check whether any zone-hour appears twice. Two events may legitimately overlap, but a boolean feature needs only one matching row. Otherwise the join would duplicate the pickup outcome.


In [ ]:
demand_df.duplicated(subset=["location_id", "hour"]).sum()

One key is repeated. I'll reduce it to one row because the current feature asks whether a post-game window exists, not how many events produced it.


In [ ]:
demand_df = demand_df.drop_duplicates(subset=["location_id", "hour"])
demand_df.duplicated(subset=["location_id", "hour"]).sum()

The cleaned keys are unique. I'll join the sports flag onto the pickup-weather table, then set unmatched rows false within the collected event definition. The pickup row count should stay the same.


In [ ]:
demand_df["is_post_game_demand"] = True

weather_joined_sports = weather_joined.merge(
    demand_df[["location_id", "hour", "is_post_game_demand"]],
    on=["location_id", "hour"],
    how="left",
)
weather_joined_sports["is_post_game_demand"] = weather_joined_sports["is_post_game_demand"].fillna(
    False
)

In [ ]:
weather_joined_sports["is_post_game_demand"].sum()

The result has 3,268 flagged zone-hours. That is not the number of games: windows can span two hourly buckets, and events can share a key. The count agrees with the cleaned event-hour table.


In [ ]:
weather_joined_sports[
    (weather_joined_sports["is_post_game_demand"])
    & (weather_joined_sports["hour"] > "2023-01-01 21:00:00")
].head(5)

### Do the flagged hours actually look busier?

The merge works, but does the window point to unusually high activity? I'll compare flagged hours with the same zone and hour a week before and after. This is a descriptive check; the later week makes it unsuitable as a real-time input.


### What happens at the same hour in nearby weeks?

The neighboring-week average gives me a reference that holds zone, weekday, and clock hour reasonably close. I'll first inspect ratios, then differences, since dividing by a tiny reference can make an ordinary pickup increase look enormous.

$$\
\text{baseline}
=
\frac{
\text{pickups one week before}
+
\text{pickups one week after}
}{2}
$$\

$$\
\text{ratio}
=
\frac{\text{post-game pickups}}{\text{baseline pickups}}
$$\


In [ ]:
comparisons = []
for _, row in weather_joined_sports[weather_joined_sports["is_post_game_demand"]].iterrows():
    week_before = row["hour"] - pd.Timedelta(days=7)
    week_after = row["hour"] + pd.Timedelta(days=7)

    before_val = weather_joined_sports[
        (weather_joined_sports["location_id"] == row["location_id"])
        & (weather_joined_sports["hour"] == week_before)
    ]["total_pickups"]
    after_val = weather_joined_sports[
        (weather_joined_sports["location_id"] == row["location_id"])
        & (weather_joined_sports["hour"] == week_after)
    ]["total_pickups"]

    if len(before_val) and len(after_val):
        baseline = (before_val.iloc[0] + after_val.iloc[0]) / 2
        if baseline > 0:
            comparisons.append(
                {
                    "location_id": row["location_id"],
                    "hour": row["hour"],
                    "event_pickups": row["total_pickups"],
                    "baseline": baseline,
                    "ratio": row["total_pickups"] / baseline,
                }
            )

comparisons_df = pd.DataFrame(comparisons)
comparisons_df["ratio"].describe()

The median ratio is about 1.52, but the mean is 3.10 and the maximum exceeds 89. That gap suggests a few extreme ratios are pulling the mean up. I'll look at their baselines before interpreting it as a typical event pattern.


In [ ]:
comparisons_df.sort_values("ratio", ascending=False).head(10)

### Investigating the Extreme Ratios

Those large ratios all come from zone 93. In one case, 134 pickups are compared with only 1.5 in the reference. The count increase may matter, but describing it as 89 times normal makes the small denominator do much of the work.


In [ ]:
comparisons_df[comparisons_df["ratio"] == 0]

In [ ]:
comparisons_df.groupby("location_id")["ratio"].median()

### Looking More Closely at Zone 93

There are also flagged hours without an increase, including one with zero pickups against a baseline of six. So the event flag is not a promise of a surge. I'll compare the venue zones separately to see how their ordinary activity differs.


In [ ]:
tlc_zones = pd.read_csv(PROJECT_ROOT / "data" / "external" / "taxi_zone_lookup.csv")

sports_zones = comparisons_df["location_id"].unique()

zones_desc = tlc_zones.loc[
    tlc_zones["LocationID"].isin(sports_zones), ["LocationID", "Borough", "Zone"]
]

zones_desc

Zone 93 covers Flushing Meadows-Corona Park. Quiet non-event hours there can make event ratios much larger than in continuously active neighborhoods. That gives me a reason to report absolute changes alongside the ratios.


### Comparing the Pickup Distribution Across Sports Zones

I'll compare ordinary pickup levels, variation, and zero shares across the sports zones. That should explain whether zone 93's unusual ratios mainly reflect its baseline or its event counts.

$$\
\text{dispersion ratio}
=
\frac{\text{variance of pickups}}
{\text{mean pickups}}
$$\


In [ ]:
def summarize_target_distribution(df, target_col="total_pickups", group_cols=None):
    if group_cols is None:
        df = df.copy()
        df["_dummy"] = 1
        group_cols = ["_dummy"]

    summary = df.groupby(group_cols).agg(
        n_rows=(target_col, "count"),
        mean=(target_col, "mean"),
        variance=(target_col, "var"),
        zero_frac=(target_col, lambda x: (x == 0).mean()),
    )
    summary["dispersion_ratio"] = summary["variance"] / summary["mean"]
    summary = summary.reset_index()

    if "_dummy" in summary.columns:
        summary = summary.drop(columns="_dummy")

    return summary

In [ ]:
result = summarize_target_distribution(weather_joined_sports, group_cols=["location_id"])
result[result["location_id"].isin([93, 181, 186, 247])]

Zone 93 averages about 19 pickups an hour, far below the other sports zones here. About 4% of its hours are zero. A citywide interpretation of the event ratio would miss that geographic difference.


In [ ]:
comparisons_df[comparisons_df["location_id"] != 93]["ratio"].describe()

### Does the post-game pattern show up near other venues?

Without zone 93, the median ratio is about 1.40 and the mean 1.57. The increase is still present, but the huge mean-median gap mostly disappears. I'll now return to zone 93 using pickup differences instead.


### Zone 93 — Using Absolute Pickup Differences

For zone 93, I'll subtract the nearby-week reference rather than divide by it. A difference keeps the result in pickups and is easier to judge when the baseline is close to zero.

$$\
\text{pickup difference}
=
\text{event-hour pickups}
-
\text{baseline pickups}
$$\


In [ ]:
comparisons_df[comparisons_df["location_id"] == 93]["event_pickups"].sub(
    comparisons_df[comparisons_df["location_id"] == 93]["baseline"]
).describe()

The median difference is about 87 pickups, with the middle half around 24–156. The large negative minimum is more surprising than another high positive value. I'll inspect those cases before deciding the window behaves consistently.


### Investigating the Negative Zone 93 Outliers

One possible explanation is another game in a reference week. That would raise the baseline and make this event look weak. I'll check the largest negative cases against nearby schedules.


In [ ]:
z93 = comparisons_df[comparisons_df["location_id"] == 93].copy()
z93["diff"] = z93["event_pickups"] - z93["baseline"]
z93.sort_values("diff").head(3)

### Another game does not explain these cases

That explanation does not hold for the three largest negative cases: no nearby games are found in those reference weeks. Other events, uncertain game-ending times, or ordinary variation may still matter. I'll see whether the pattern changes by hour.


In [ ]:
suspect_dates = ["2022-08-25", "2022-09-15", "2022-09-16"]

for d in suspect_dates:
    event_time = pd.Timestamp(d + " 23:00:00")
    week_before = event_time - pd.Timedelta(days=7)
    week_after = event_time + pd.Timedelta(days=7)

    nearby = games_df[
        (games_df["location_id"] == 93)
        & (
            games_df["start_time_et"]
            .dt.tz_localize(None)
            .between(week_before - pd.Timedelta(hours=6), week_before + pd.Timedelta(hours=6))
            | games_df["start_time_et"]
            .dt.tz_localize(None)
            .between(week_after - pd.Timedelta(hours=6), week_after + pd.Timedelta(hours=6))
        )
    ]
    print(d, "-> nearby control-week games:", len(nearby))

In [ ]:
z93 = comparisons_df[comparisons_df["location_id"] == 93].copy()
z93["hour_of_day"] = z93["hour"].dt.hour
z93["diff"] = z93["event_pickups"] - z93["baseline"]
z93.groupby("hour_of_day")["diff"].describe()

### The pattern changes by hour

The strongest median increases are around 9–10 PM, and the pattern is much weaker by 11 PM. That helps explain why one broad post-event label may not be equally useful at every hour. It still does not give me exact ending times for individual games.


In [ ]:
weather_joined_sports.shape

In [ ]:
weather_joined_sports.to_parquet(
    PROJECT_ROOT / "data" / "gold" / "zone_hour_pickups_weather.parquet"
)

## Historical Concert Events Data from Setlist.fm

I'll use setlist.fm for historical concerts after the Ticketmaster coverage check. First I need the right venue records, because a correct event date at the wrong venue would create a misleading local feature.


In [ ]:
headers = {"x-api-key": SETLISTFM_API_KEY_1, "Accept": "application/json"}

response = requests.get(
    "https://api.setlist.fm/rest/1.0/search/venues",
    headers=headers,
    params={"name": "Madison Square Garden"},
)
# print(response.json())

In [ ]:
data = response.json()
venues_example_df = pd.json_normalize(data["venue"])
venues_example_df.columns

In [ ]:
venue_schema = json_schema(data["venue"][0])

print(json.dumps(venue_schema, indent=4))

In [ ]:
venues_display = venues_example_df[
    [
        "id",
        "name",
        "city.name",
        "city.state",
        "city.stateCode",
        "city.coords.lat",
        "city.coords.long",
        "city.country.name",
        "url",
    ]
].copy()

venues_display.columns = [
    "venue_id",
    "venue_name",
    "city",
    "state",
    "state_code",
    "latitude",
    "longitude",
    "country",
    "setlist_url",
]

venues_display

A name match is not enough. Madison Square Garden appears in more than one city in the search results. I'll inspect city and state before accepting a venue ID.


In [ ]:
response = requests.get(
    "https://api.setlist.fm/rest/1.0/search/setlists",
    headers={"x-api-key": SETLISTFM_API_KEY_1, "Accept": "application/json"},
    params={"cityName": "Brooklyn", "year": 2023},  # MSG's real ID
)
print(response.json())

### Building the NYC Concert Venue List

I'll focus on a selected group of major venues rather than try to cover every live performance in NYC. That keeps the first feature manageable, but any result needs to be interpreted within that selected coverage.


### Assigning Each Venue to a TLC Taxi Zone

I'll map each venue's coordinates to a TLC polygon before linking source IDs. This keeps the geographic decision separate from matching names in the API.


In [ ]:
concert_venues = pd.DataFrame(
    {
        "venue_name": [
            "Madison Square Garden",
            "Barclays Center",
            "Yankee Stadium",
            "Citi Field",
            "Radio City Music Hall",
            "Apollo Theater",
            "Kings Theatre",
            "Beacon Theatre",
            "Terminal 5",
            "Carnegie Hall",
            "Forest Hills Stadium",
        ],
        "latitude": [
            40.7503,
            40.6827,
            40.8296,
            40.7570,
            40.7597,
            40.8100,
            40.6458,
            40.7806,
            40.7697,
            40.7650,
            40.7196,
        ],
        "longitude": [
            -73.9933,
            -73.9752,
            -73.9262,
            -73.8458,
            -73.9792,
            -73.9500,
            -73.9575,
            -73.9811,
            -73.9928,
            -73.9800,
            -73.8487,
        ],
    }
)
concert_venues

I'll check the venue points against the zone boundaries rather than assign IDs from neighborhood names.


In [ ]:
points = []
for i in range(len(concert_venues)):
    point = Point(concert_venues.iloc[i]["longitude"], concert_venues.iloc[i]["latitude"])
    points.append(point)

matching_location_ids = []
for p in range(len(points)):
    match = zones_gdf[zones_gdf.contains(points[p])]
    matching_location_ids.append(
        [
            concert_venues["venue_name"].iloc[p],
            match["locationid"].iloc[0],
            match["borough"].iloc[0],
            match["zone"].iloc[0],
        ]
    )

concert_venues_with_zones = pd.DataFrame(
    matching_location_ids, columns=["venue_name", "location_id", "borough", "zone"]
)
concert_venues_with_zones["location_id"] = concert_venues_with_zones["location_id"].astype(int)
concert_venues_with_zones

All 11 coordinate points find a zone in this step. For example, Madison Square Garden maps to zone 186. Now I need the corresponding setlist.fm records for those physical locations.


### Resolving the Correct Setlist.fm Venue IDs

I'll search for the source IDs and review the returned locations. Venue aliases and repeated names mean the first result is not always the intended one.


In [ ]:
for name in concert_venues_with_zones["venue_name"]:
    # if name in ['Madison Square Garden', 'Kings Theatre']:  # already have these
    #     continue
    response = requests.get(
        "https://api.setlist.fm/rest/1.0/search/venues",
        headers={"x-api-key": SETLISTFM_API_KEY_1, "Accept": "application/json"},
        params={"name": name},
    )
    result = response.json()
    print(f"--- {name} ---")
    for v in result.get("venue", [])[:5]:
        city = v.get("city", {})
        time.sleep(1)
        print(f"  {v.get('name')} | id={v.get('id')} | {city.get('name')}, {city.get('stateCode')}")
    print()

In [ ]:
remaining_venues = [
    "Apollo Theater",
    "Beacon Theatre",
    "Terminal 5",
    "Carnegie Hall",
    "Forest Hills Stadium",
]

for name in remaining_venues:
    response = requests.get(
        "https://api.setlist.fm/rest/1.0/search/venues",
        headers={"x-api-key": SETLISTFM_API_KEY_1, "Accept": "application/json"},
        params={"name": name},
    )
    result = response.json()
    print(f"--- {name} ---")
    for v in result.get("venue", [])[:5]:
        city = v.get("city", {})
        print(f"  {v.get('name')} | id={v.get('id')} | {city.get('name')}, {city.get('stateCode')}")
    print()
    time.sleep(2)  # stay safely under 2 requests/second

# Kings Theatre — try under its other known name
for search_name in ["Paramount", "Brooklyn Paramount"]:
    response = requests.get(
        "https://api.setlist.fm/rest/1.0/search/venues",
        headers={"x-api-key": SETLISTFM_API_KEY_1, "Accept": "application/json"},
        params={"name": search_name},
    )
    result = response.json()
    print(f"--- searching '{search_name}' ---")
    for v in result.get("venue", [])[:5]:
        city = v.get("city", {})
        print(f"  {v.get('name')} | id={v.get('id')} | {city.get('name')}, {city.get('stateCode')}")
    print()
    time.sleep(2)

The searches confirm the problem: names such as Beacon Theatre return records outside the target location. I'll keep only reviewed matches and retain the lookup so corrections do not require editing several download functions.


In [ ]:
for search_name in ["Paramount", "Brooklyn Paramount"]:
    response = requests.get(
        "https://api.setlist.fm/rest/1.0/search/venues",
        headers={"x-api-key": SETLISTFM_API_KEY_1, "Accept": "application/json"},
        params={"name": "Kings Theater", "cityName": "Brooklyn"},
    )
    result = response.json()
    print(f"--- searching '{search_name}' ---")
    for v in result.get("venue", [])[:5]:
        city = v.get("city", {})
        print(f"  {v.get('name')} | id={v.get('id')} | {city.get('name')}, {city.get('stateCode')}")
    print()
    time.sleep(2)

The Kings Theatre search leads to a Brooklyn record through Paramount-related names. That needs more than a plausible alias before I rely on it. I'll preserve the current record here, but the physical address, coordinates, and historical names need to be checked against the intended venue.


### Building the Final Concert Venue Reference Table

I'll put the selected source IDs and TLC zones into one reference table. It becomes the place to review uncertain matches before those decisions spread into every downloaded year.


In [ ]:
concert_venues_final = [
    {"venue_name": "Madison Square Garden", "setlistfm_id": "23d63cc7", "location_id": 186},
    {
        "venue_name": "The Theater at Madison Square Garden",
        "setlistfm_id": "7bd62a20",
        "location_id": 186,
    },
    {"venue_name": "Barclays Center", "setlistfm_id": "2bd77066", "location_id": 181},
    {"venue_name": "Yankee Stadium", "setlistfm_id": "5bd38390", "location_id": 247},
    {"venue_name": "Citi Field", "setlistfm_id": "63d1eed3", "location_id": 93},
    {"venue_name": "Radio City Music Hall", "setlistfm_id": "23d624c7", "location_id": 161},
    {"venue_name": "Kings Theatre", "setlistfm_id": "73d45aa1", "location_id": 85},
    {"venue_name": "Beacon Theatre", "setlistfm_id": "bd6c9ba", "location_id": 239},
    {"venue_name": "Terminal 5", "setlistfm_id": "6bd6227a", "location_id": 50},
    {"venue_name": "Carnegie Hall", "setlistfm_id": "7bd63668", "location_id": 163},
    {"venue_name": "Forest Hills Stadium", "setlistfm_id": "1bd26dfc", "location_id": 95},
    {"venue_name": "Forest Hills Tennis Stadium", "setlistfm_id": "13d26df1", "location_id": 95},
]

With the reference assembled, I can use the same mapping for every yearly request.


### Downloading Historical Concert Data from Setlist.fm

I'll collect one venue-year at a time and save the raw response. If a request fails or a venue needs correction, I can revisit that piece without downloading the entire history again.


In [ ]:
def fetch_setlistfm_venue(venue_name, venue_id, year, api_key):
    """
    Fetch all setlists for one venue in one calendar year from the
    setlist.fm API, handling pagination, and save the combined result
    to disk. Skips the fetch entirely if that file already exists —
    same idempotent pattern as fetch_ticketmaster_events and
    fetch_espn_schedule.

    Args:
        venue_name : str
            Human-readable venue name, used to build the saved filename.
        venue_id : str
            The venue's real setlist.fm ID (from /search/venues).
        year : int
            Calendar year to fetch setlists for.
        api_key : str
            setlist.fm API key, sent via the x-api-key header.

    Returns:
        None
            It only writes a JSON file to EVENTS_DIR; does not return the data
            directly (same as other fetch functions this
            project uses).
    """
    EVENTS_DIR = PROJECT_ROOT / "data" / "external" / "events" / "concerts"
    EVENTS_DIR.mkdir(exist_ok=True, parents=True)
    safe_name = venue_name.lower().replace(" ", "_")
    filepath = EVENTS_DIR / f"{safe_name}_{year}_setlists.json"

    if filepath.exists():
        print(f"Already cached, skipping: {venue_name}, {year}")
        return

    headers = {"x-api-key": api_key, "Accept": "application/json"}
    all_setlists = []
    page = 1

    while True:
        response = requests.get(
            "https://api.setlist.fm/rest/1.0/search/setlists",
            headers=headers,
            params={"venueId": venue_id, "year": year, "p": page},
        )
        result = response.json()
        if "setlist" not in result:
            break
        current_page_shows = result["setlist"]
        all_setlists.extend(current_page_shows)
        if len(current_page_shows) < result["itemsPerPage"]:
            break
        page = page + 1
        time.sleep(1)

    with open(filepath, "w") as f:
        json.dump({"setlist": all_setlists}, f)

    print(f"Downloaded {venue_name}'s {year} season")

    # print(all_setlists)

In [ ]:
for venue in concert_venues_final:
    for year in range(2021, 2027):
        fetch_setlistfm_venue(
            venue_name=venue["venue_name"],
            venue_id=venue["setlistfm_id"],
            year=year,
            api_key=SETLISTFM_API_KEY_1,
        )
        time.sleep(1)

### Verifying the Historical Concert Download

Before parsing, I'll compare the saved files with the expected venue-year combinations. Missing files should be a collection problem, not silently become dates without concerts.


In [ ]:
concert_files = list(
    (PROJECT_ROOT / "data" / "external" / "events" / "concerts").glob("*_setlists.json")
)
print(len(concert_files), "files")

There are 12 source IDs across six years, giving 72 files, and all 72 are present. Source IDs are not automatically distinct physical venues, so the reference review remains important even when the file count is right.

$$\
2021,\ 2022,\ 2023,\ 2024,\ 2025,\ 2026
$$\

$$\
12 \times 6 = 72
$$\


### Is one setlist the same as one concert?

Now I'll inspect what one source record actually represents. That will decide whether counting rows is meaningful for the feature I want.


In [ ]:
with open(
    PROJECT_ROOT
    / "data"
    / "external"
    / "events"
    / "concerts"
    / "madison_square_garden_2023_setlists.json"
) as f:
    data = json.load(f)
print(len(data["setlist"]), "shows found")

The Madison Square Garden file has 233 setlists for 2023. That does not mean 233 concerts: an opening act and a headliner can have separate records for the same evening. I'll work toward a zone-date signal instead of calling every artist setlist a separate event.

$$\
\text{Setlist.fm record} \neq \text{necessarily one unique concert event}
$$\


In [ ]:
with open(
    PROJECT_ROOT
    / "data"
    / "external"
    / "events"
    / "concerts"
    / "madison_square_garden_2023_setlists.json"
) as f:
    data = json.load(f)
print(data["setlist"][0])

In [ ]:
def show_json_schema(obj):
    if isinstance(obj, dict):
        return {key: show_json_schema(value) for key, value in obj.items()}

    elif isinstance(obj, list):
        if len(obj) == 0:
            return ["empty_list"]

        return [show_json_schema(obj[0])]

    else:
        return type(obj).__name__

In [ ]:
setlist_schema = show_json_schema(data["setlist"][0])

print(json.dumps(setlist_schema, indent=4))

### Inspecting the Setlist.fm Event Structure

The record has a date, artist, venue, and optional notes. What I still need for a true post-concert window is a reliable ending time. I'll look for that before choosing a fixed evening assumption.

$$\
\text{eventDate} = \text{calendar date only}
$$\

$$\
(\text{location\_id},\ \text{hour})
$$\


In [ ]:
with open(
    PROJECT_ROOT
    / "data"
    / "external"
    / "events"
    / "concerts"
    / "madison_square_garden_2023_setlists.json"
) as f:
    data = json.load(f)

shows_with_info = [s for s in data["setlist"] if "info" in s]
shows_with_times = [s for s in shows_with_info if "time" in s["info"] or "showtime" in s["info"]]

print(f"{len(data['setlist'])} total shows")
print(f"{len(shows_with_info)} have an 'info' field at all")
print(f"{len(shows_with_times)} of those actually mention Doors/Show times")

if shows_with_times:
    print()
    print("Example:", shows_with_times[0]["info"])

### Deeper Search for Timing Information in the `info` Field

The first keyword search finds little useful timing information. I'll search the notes for clock-like values as well, then read their context rather than treat any time string as a show ending.


In [ ]:
import re

keywords = [
    "door",
    "doors",
    "show",
    "start",
    "started",
    "stage",
    "on stage",
    "curfew",
    "early show",
    "late show",
    "matinee",
    "pm",
    "am",
]

matches = []

concert_dir = PROJECT_ROOT / "data" / "external" / "events" / "concerts"

for filepath in concert_dir.glob("*_setlists.json"):
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    for event in data.get("setlist", []):
        info = event.get("info", "")

        if not info:
            continue

        info_lower = info.lower()

        found_keywords = [keyword for keyword in keywords if keyword in info_lower]

        if found_keywords:
            matches.append(
                {
                    "file": filepath.name,
                    "event_date": event.get("eventDate"),
                    "artist": event.get("artist", {}).get("name"),
                    "venue": event.get("venue", {}).get("name"),
                    "keywords": found_keywords,
                    "info": info,
                }
            )

In [ ]:
timing_keyword_df = pd.DataFrame(matches)

print("Matches found:", len(timing_keyword_df))

timing_keyword_df.head(20)

In [ ]:
time_pattern = re.compile(
    r"\b(?:1[0-2]|0?[1-9])" r"(?::[0-5][0-9])?" r"\s?(?:am|pm|a\.m\.|p\.m\.)\b", flags=re.IGNORECASE
)

In [ ]:
time_matches = []

for filepath in concert_dir.glob("*_setlists.json"):
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)

    for event in data.get("setlist", []):
        info = event.get("info", "")

        if not info:
            continue

        times_found = time_pattern.findall(info)

        if times_found:
            time_matches.append(
                {
                    "file": filepath.name,
                    "event_date": event.get("eventDate"),
                    "artist": event.get("artist", {}).get("name"),
                    "venue": event.get("venue", {}).get("name"),
                    "times_found": times_found,
                    "info": info,
                }
            )

In [ ]:
time_matches_df = pd.DataFrame(time_matches)

print("Records containing clock-like times:", len(time_matches_df))

time_matches_df.head(30)

The matches do not consistently mean what I need. A time can refer to an artist's set, a curfew, or an interruption. Combining those into one supposedly exact ending-time field would add false precision.

$$\
\text{clock time found in text}
\neq
\text{verified concert start/end time}
$$\


In [ ]:
with open(
    PROJECT_ROOT / "data" / "external" / "events" / "concerts" / "kings_theatre_2023_setlists.json"
) as f:
    data2 = json.load(f)

shows_with_info2 = [s for s in data2["setlist"] if "info" in s]
shows_with_times2 = [s for s in shows_with_info2 if "Doors" in s["info"] or "Show" in s["info"]]
print(f"{len(data2['setlist'])} total, {len(shows_with_times2)} with times")

In [ ]:
total_records = 0
records_with_info = 0
records_with_clock_time = 0

for filepath in concert_dir.glob("*_setlists.json"):
    with open(filepath, encoding="utf-8") as f:
        file_data = json.load(f)

    for event in file_data.get("setlist", []):
        total_records += 1

        info = event.get("info", "")

        if info:
            records_with_info += 1

        if info and time_pattern.search(info):
            records_with_clock_time += 1


clock_time_coverage = records_with_clock_time / total_records * 100 if total_records > 0 else 0

info_coverage = records_with_info / total_records * 100 if total_records > 0 else 0

print(f"Total Setlist.fm records: {total_records:,}")
print(f"Records with an info field: {records_with_info:,} ({info_coverage:.2f}%)")
print(
    f"Records with a clock-like time in info: "
    f"{records_with_clock_time:,} ({clock_time_coverage:.4f}%)"
)

### Measuring How Much Timing Information Is Actually Available

Only five of the 4,620 records contain a clock-like time in this check. Even those are not consistent event endings. I'll use the stronger date and venue information rather than build the feature around a handful of ambiguous notes.

$$\
\frac{827}{4620} \times 100 \approx 17.90\%
$$\

$$\
\frac{5}{4620} \times 100 \approx 0.1082\%
$$\

$$\
\text{venue} + \text{event date}
$$\


### Trying approximate post-concert windows

Since exact endings are unavailable, I'll test a few late-evening windows meant to approximate when concertgoers leave. The data can tell me whether those flags help prediction, but not whether every concert finished before the flagged period.

$$\
\text{game start}
\rightarrow
\text{game end}
\rightarrow
\text{post-game pickup window}
$$\

$$\
\text{venue}
+
\text{event date}
$$\

$$\
21:00 \leq \text{hour} < 23:00
$$\

$$\
22:00 \leq \text{hour} < 24:00
$$\

$$\
\text{Dataset A}
\quad vs \quad
\text{Dataset B}
\quad vs \quad
\text{Dataset C}
$$\


### Combining the Concert Files Into One Event-Date Table

I'll combine the saved files into a smaller table of event dates and venue zones. Songs and artist details are useful for the source, but not necessary for this first pickup indicator.

$$\
\text{location\_id} + \text{event\_date}
$$\

$$\
(\text{location\_id},\ \text{event\_date})
$$\

$$\
(\text{location\_id}=186,\ \text{event\_date}=\text{concert date})
$$\


In [ ]:
concert_rows = []
for venue in concert_venues_final:
    for year in range(2021, 2027):
        safe_name = venue["venue_name"].lower().replace(" ", "_")
        filepath = (
            PROJECT_ROOT
            / "data"
            / "external"
            / "events"
            / "concerts"
            / f"{safe_name}_{year}_setlists.json"
        )

        with open(filepath) as f:
            data = json.load(f)
            for event in data["setlist"]:
                eventDate = event["eventDate"]
                event_info = {"location_id": venue["location_id"], "event_date": eventDate}
                concert_rows.append(event_info)

concerts_df = pd.DataFrame(concert_rows)

concerts_df["event_date"] = pd.to_datetime(concerts_df["event_date"], format="%d-%m-%Y")

concerts_df = concerts_df.drop_duplicates(subset=["location_id", "event_date"])

### Parsing the Concert Dates Correctly

The dates use day-month-year. I'll specify `%d-%m-%Y` explicitly so an ambiguous value cannot silently switch month and day during parsing.

$$\
05\text{-}06\text{-}2023
=
\text{June 5, 2023}
$$\

$$\
\text{day-month-year}
$$\


In [ ]:
concerts_df.head()

### Final Concert Event Table

After removing repeated zone-dates, one row means at least one collected concert-related record in that zone on that date. This avoids multiplying the signal because several performers have setlists from the same evening.

$$\
\text{one TLC taxi zone on one date with at least one concert-related event}
$$\

$$\
(\text{location\_id}=186,\ \text{event\_date}=2021\text{-}12\text{-}20)
$$\

$$\
\text{Was there at least one concert event in this TLC zone on this date?}
$$\

$$\
\text{How many separate concerts occurred in this zone?}
$$\


### Creating the Hourly Concert-Window Feature

The date-level signal now needs to become hourly. I'll expand it into candidate windows and pay attention to the following date when the period crosses midnight.

$$\
\text{location\_id}
+
\text{event\_date}
$$\

$$\
\text{start\_hour}=21
$$\

$$\
\text{end\_hour}=23
$$\

$$\
21:00 \leq \text{hour} < 23:00
$$\

$$\
(\text{location\_id},\ \text{date})
$$\

$$\
\text{is\_concert\_window}
=
\text{concert day}
\land
\text{hour inside selected window}
$$\

$$\
\text{is\_concert\_window}
\in
\{\text{False},\text{True}\}
$$\

$$\
\text{start\_hour} \leq \text{hour\_of\_day} < \text{end\_hour}
$$\


### Applying Two Different Concert-Time Assumptions

I'll begin with two plausible evening definitions and keep everything else the same.


In [ ]:
def add_concert_window_flag(df, concerts_df, start_hour, end_hour):
    """
    Flag zone-hour rows that fall within a fixed evening window on a
    day when that zone had a concert. Unlike the sports post-game
    feature, this has no per-show start time to work from — setlist.fm
    genuinely doesn't provide one (confirmed against the official
    schema) — so this uses a fixed clock-time window instead of a
    computed one.

    Args:
        df : the zone-hour dataframe (weather_joined or similar)
        concerts_df : location_id + event_date, one row per concert-day
        start_hour, end_hour : the fixed window, e.g. 21 and 23 for 9-11pm

    Returns:
        df with a new boolean column added
    """
    df = df.copy()
    concert_dates = concerts_df[["location_id", "event_date"]].drop_duplicates()
    concert_dates = concert_dates.rename(columns={"event_date": "concert_date"})

    df["date_only"] = df["hour"].dt.date
    concert_dates["concert_date"] = concert_dates["concert_date"].dt.date

    merged = df.merge(
        concert_dates,
        left_on=["location_id", "date_only"],
        right_on=["location_id", "concert_date"],
        how="left",
    )

    is_concert_day = merged["concert_date"].notna()
    is_in_window = (df["hour_of_day"] >= start_hour) & (df["hour_of_day"] < end_hour)

    df["is_concert_window"] = (is_concert_day & is_in_window).values
    df = df.drop(columns=["date_only"])
    return df

A covers 9–11 PM, while B covers 10 PM–midnight. Neither is a known end time. They are alternative ways to represent the possible post-concert period, and the model comparisons will help decide whether either is useful.

$$
\text{start\_hour} = 21
$$

$$
\text{end\_hour} = 23
$$

$$
21 \leq \text{hour\_of\_day} < 23
$$

$$
\text{start\_hour} = 22
$$

$$
\text{end\_hour} = 24
$$

$$
22 \leq \text{hour\_of\_day} < 24
$$

$$
\text{Dataset A}
=
\text{base data}
+
\text{concert window from 9 PM to 11 PM}
$$

$$
\text{Dataset B}
=
\text{base data}
+
\text{concert window from 10 PM to midnight}
$$


In [ ]:
def canonical_model_order(df):
    return df.sort_values(["hour", "location_id"]).reset_index(drop=True).copy()

In [ ]:
# Create dataset A: concert window 9 PM - 11 PM
weather_joined_sports_concerts_21_23 = add_concert_window_flag(
    weather_joined_sports, concerts_df, start_hour=21, end_hour=23
)

# Create dataset B: concert window 10 PM - midnight
weather_joined_sports_concerts_22_24 = add_concert_window_flag(
    weather_joined_sports, concerts_df, start_hour=22, end_hour=24
)

# Put both datasets into the same deterministic row order
weather_joined_sports_concerts_21_23 = canonical_model_order(weather_joined_sports_concerts_21_23)

weather_joined_sports_concerts_22_24 = canonical_model_order(weather_joined_sports_concerts_22_24)

print("A concert rows:", weather_joined_sports_concerts_21_23["is_concert_window"].sum())

print("B concert rows:", weather_joined_sports_concerts_22_24["is_concert_window"].sum())

In [ ]:
# weather_joined_sports_concerts_21_23 = add_concert_window_flag(weather_joined_sports, concerts_df,
#                                      start_hour=21, end_hour=23)
# weather_joined_sports_concerts_22_24 = add_concert_window_flag(weather_joined_sports, concerts_df,
#                                      start_hour=22, end_hour=24)

### Checking How Many Zone-Hour Rows Were Flagged

I'll count the flagged rows before fitting. This catches mistakes in the expansion without having to diagnose them from model scores later.


In [ ]:
print(weather_joined_sports_concerts_21_23["is_concert_window"].sum())
print(weather_joined_sports_concerts_22_24["is_concert_window"].sum())

Both versions flag 4,372 rows. That makes sense because both windows have two hours for the same eligible zone-dates. Their counts agree, even though they point to different clock hours.

$$
21 \leq \text{hour\_of\_day} < 23
$$

$$
22 \leq \text{hour\_of\_day} < 24
$$

| Hour | Dataset A | Dataset B |
|---|---:|---:|
| 9 PM | True | False |
| 10 PM | True | True |
| 11 PM | False | True |

$$
\text{how many rows are marked as concerts}
$$

$$
\text{which hours are marked as concert-window pickups}
$$


In [ ]:
unique_concert_zone_dates = len(concerts_df[["location_id", "event_date"]].drop_duplicates())

expected_flagged_hours = unique_concert_zone_dates * 2

print(f"Unique concert zone-dates: {unique_concert_zone_dates:,}")
print(f"Expected flagged hours for a 2-hour window: {expected_flagged_hours:,}")
print("Actual flagged hours:", int(weather_joined_sports_concerts_21_23["is_concert_window"].sum()))

In [ ]:
print("Concert date range:")
print(concerts_df["event_date"].min())
print(concerts_df["event_date"].max())

after_data_end = concerts_df[concerts_df["event_date"] > "2026-06-30"]

print()
print("Concert zone-dates after 2026-06-30:", len(after_data_end))

after_data_end.head()

### Explaining the Difference Between Expected and Actual Concert Flags

The first expected count was too high because some concert zone-dates fall outside the pickup period. The 132-row difference corresponds to 66 two-hour dates. I'll align coverage before treating it as a failed join.

$$
2252 \times 2 = 4504
$$

$$
4372
$$

$$
4504 - 4372 = 132
$$

$$
\frac{132}{2} = 66
$$

$$
2026\text{-}08\text{-}13
$$

$$
2026\text{-}06\text{-}30
$$

$$
2252 - 66 = 2186
$$

$$
2186 \times 2 = 4372
$$


### A note about the modeling notes

There are additional notes in `reports/MODELING_EXPLAINED.md`, but that directory is ignored by Git. I'll keep the reasoning needed to follow these comparisons in the notebook itself.


### Setting Up Dataset A for the First Model Comparison

I'll start fitting with Dataset A, then use the same procedure for the alternatives.


In [ ]:
feature_cols = [
    "location_id",
    "temperature_f",
    "precipitation_in",
    "rain_in",
    "snowfall_in",
    "wind_speed_mph",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "month",
    "is_weekday_rush_hour",
    "is_weekend_night",
    "is_holiday",
    "is_unofficial_holiday",
    "is_post_game_demand",
    "is_concert_window",
]
target_col = "total_pickups"

train_A = weather_joined_sports_concerts_21_23[
    (weather_joined_sports_concerts_21_23["hour"] < "2026-01-01")
]

test_A = weather_joined_sports_concerts_21_23[
    weather_joined_sports_concerts_21_23["hour"] >= "2026-01-01"
]

X_train_A = train_A[feature_cols]
y_train_A = train_A[target_col]

X_test_A = test_A[feature_cols]
y_test_A = test_A[target_col]

### Using a Time-Based Train/Test Split

The first model uses zone, weather, calendar, and sports information alongside the concert flag. This remains a retrospective feature experiment with historical target-hour weather. It helps choose a signal to investigate, but it is not the later forecast-origin evaluation.

$$
\text{total pickups}
=
f(
\text{location},
\text{weather},
\text{calendar},
\text{sports},
\text{concerts}
)
$$

$$
2021\text{-}01\text{-}01
\quad \text{through} \quad
2025\text{-}12\text{-}31
$$

$$
2026\text{-}01\text{-}01
$$

$$
\text{historical data}
\rightarrow
\text{train model}
\rightarrow
\text{predict future pickups}
$$


In [ ]:
print(X_train_A.shape, X_test_A.shape)
print(y_train_A.shape, y_test_A.shape)

### Training the Baseline Model for Dataset A

I'll use LightGBM as a common model for the window comparisons. The important part here is holding the procedure fixed, not claiming that this initial configuration is the best forecasting model.

$$
\hat{y}
=
f_1(X)
+
f_2(X)
+
\cdots
+
f_M(X)
$$

$$
\text{concert feature}
$$

$$
\text{different model settings}
$$

$$
\text{same model}
+
\text{same train/test period}
+
\text{same features}
+
\text{different concert assumption}
$$

$$
X_{\text{test}}
\rightarrow
\text{trained LightGBM model}
\rightarrow
\hat{y}_{\text{test}}
$$


In [ ]:
model_A = LGBMRegressor(random_state=42, deterministic=True, force_row_wise=True)
model_A.fit(X_train_A, y_train_A)

In [ ]:
y_pred_A = model_A.predict(X_test_A)

### First Results for Dataset A

I'll first score all evaluation rows, then look at the much smaller concert subset.


In [ ]:
mae_A = mean_absolute_error(y_test_A, y_pred_A)
rmse_A = np.sqrt(mean_squared_error(y_test_A, y_pred_A))
r2_A = r2_score(y_test_A, y_pred_A)

print(f"MAE:  {mae_A:.2f}")
print(f"RMSE: {rmse_A:.2f}")
print(f"R2:   {r2_A:.4f}")

The pooled MAE is about 41.6 pickups, while RMSE is 66.1. Some larger misses are pulling RMSE upward. Those numbers give a starting point, but a feature active in only a few hours could be useful without moving the pooled score very far.

$$
MAE
=
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat{y}_i|
$$

$$
RMSE
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y}_i)^2
}
$$

$$
MAE = 41.61
$$

$$
RMSE = 66.06
$$

$$
R^2 = 0.6949
$$

$$
69.49\%
$$

$$
\text{is\_concert\_window} = \text{True}
$$


In [ ]:
for run in range(3):
    model_test = LGBMRegressor(random_state=42, deterministic=True, force_row_wise=True)

    model_test.fit(X_train_A, y_train_A)

    pred_test = model_test.predict(X_test_A)

    mae = mean_absolute_error(y_test_A, pred_test)
    rmse = np.sqrt(mean_squared_error(y_test_A, pred_test))
    r2 = r2_score(y_test_A, pred_test)

    print(f"Run {run + 1}: " f"MAE={mae:.6f}, " f"RMSE={rmse:.6f}, " f"R2={r2:.6f}")

### Evaluating Dataset A Specifically During Concert-Window Hours

I'll isolate concert-window rows next. Otherwise thousands of ordinary hours could hide whether the model is doing anything useful during the periods the flag was built for.

$$
\text{is\_concert\_window}
=
\text{True}
$$

$$
\text{all 2026 test observations}
$$

$$
\text{2026 observations occurring during the selected concert window}
$$

$$
y_i
\quad \text{vs.} \quad
\hat{y}_i
$$


In [ ]:
concert_mask_A = test_A["is_concert_window"]

y_test_concert_A = y_test_A[concert_mask_A]
y_pred_concert_A = y_pred_A[concert_mask_A]

### Dataset A Results During Concert-Window Hours

Here are the results for A's flagged hours.


In [ ]:
mae_concert_A = mean_absolute_error(y_test_concert_A, y_pred_concert_A)
rmse_concert_A = np.sqrt(mean_squared_error(y_test_concert_A, y_pred_concert_A))
r2_concert_A = r2_score(y_test_concert_A, y_pred_concert_A)

print(f"MAE concert_A:   {mae_concert_A}")
print(f"RMSE concert_A: {rmse_concert_A}")
print(f"R2 concert_A:    {r2_concert_A}")

Errors are much larger within those hours: MAE is about 88.5 and RMSE 119.0. That suggests a harder, possibly higher-volume subset. To compare windows fairly, I'll need to evaluate alternatives on common hours rather than each one's own selected rows.

$$
41.61
\rightarrow
88.50
$$

$$
66.06
\rightarrow
118.99
$$

$$
0.6949
\rightarrow
0.5944
$$

$$
\text{high concert-window error}
\neq
\text{concert feature is ineffective}
$$


### Building One Reusable Comparison Function

I'll put the repeated fitting and scoring steps into `run_comparison()`. That makes it easier to keep model settings and row handling consistent while changing the concert feature.

$$
\text{Dataset}
\rightarrow
\text{same training procedure}
\rightarrow
\text{same evaluation procedure}
\rightarrow
\text{comparable metrics}
$$

$$
21 \leq \text{hour} < 23
$$

$$
22 \leq \text{hour} < 24
$$

$$
\text{eval\_mask}
=
\text{A window}
\lor
\text{B window}
$$

$$
A = \{21,22\}
$$

$$
B = \{22,23\}
$$

$$
A \cup B = \{21,22,23\}
$$

$$
\text{A with 9--11 PM feature}
$$

$$
\text{vs.}
$$

$$
\text{B with 10 PM--midnight feature}
$$

$$
\text{vs.}
$$

$$
\text{C with no concert feature}
$$

$$
\text{error with concert information}
\quad \text{vs.} \quad
\text{error without concert information}
$$

$$
e_i = y_i - \hat{y}_i
$$

$$
y_i > \hat{y}_i
$$

$$
y_i < \hat{y}_i
$$


In [ ]:
def run_comparison(
    df, label, feature_cols, eval_mask, target_col="total_pickups", objective="regression"
):
    """
    Train a quick LightGBM model on df, evaluate on both the full 2026
    test set and the fixed concert-window evaluation slice. Prints
    MAE/RMSE/R2 for both. Returns a dict of the concert-window metrics,
    for later comparison across datasets.
    """

    df_train = df[df["hour"] < "2026-01-01"]
    df_test = df[df["hour"] >= "2026-01-01"]

    X_train = df_train[feature_cols]
    y_train = df_train[target_col]

    X_test = df_test[feature_cols]
    y_test = df_test[target_col]

    model = LGBMRegressor(
        random_state=42, objective=objective, deterministic=True, force_row_wise=True
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    test_mask = eval_mask.loc[df_test.index]

    y_test_concert = y_test[test_mask]
    y_pred_concert = y_pred[test_mask.values]

    mae_concert = mean_absolute_error(y_test_concert, y_pred_concert)
    rmse_concert = np.sqrt(mean_squared_error(y_test_concert, y_pred_concert))
    r2_concert = r2_score(y_test_concert, y_pred_concert)
    residuals = y_test_concert - y_pred_concert

    location_id_concert = df_test["location_id"][test_mask]

    return {
        "label": label,
        "mae": mae_concert,
        "rmse": rmse_concert,
        "r2": r2_concert,
        "residuals": residuals,
        "y_test": y_test_concert,
        "y_pred": y_pred_concert,
        "location_id": location_id_concert,
    }

### Comparing the Two Concert Windows Against the No-Concert Baseline

Now I can compare A, B, and the no-concert control under the same procedure.


In [ ]:
weather_joined_sports = canonical_model_order(weather_joined_sports)

In [ ]:
eval_mask = (
    weather_joined_sports_concerts_21_23["is_concert_window"]
    | weather_joined_sports_concerts_22_24["is_concert_window"]
)

hour23_only_mask = (
    weather_joined_sports_concerts_22_24["is_concert_window"]
    & ~weather_joined_sports_concerts_21_23["is_concert_window"]
)

feature_cols_C = [c for c in feature_cols if c != "is_concert_window"]

result_A = run_comparison(
    weather_joined_sports_concerts_21_23, "A (9-11pm)", feature_cols, eval_mask
)
result_B = run_comparison(
    weather_joined_sports_concerts_22_24, "B (10pm-midnight)", feature_cols, eval_mask
)
result_C = run_comparison(weather_joined_sports, "C (no concerts)", feature_cols_C, eval_mask)

pd.DataFrame([result_A, result_B, result_C])

Sorting by hour and zone removes an input-order difference between runs. B performs better than A on the shared comparison. I'll look at the hour where their definitions differ most, rather than stop at that overall ranking.

| Dataset | Concert Window | MAE | RMSE | R² |
|---|---|---:|---:|---:|
| A | 9 PM–11 PM | 102.29 | 139.54 | 0.39 |
| B | 10 PM–midnight | **95.64** | **131.75** | **0.46** |
| C | No concert feature | 119.48 | 166.36 | 0.14 |


## What does the 11 PM hour add?

A and B both flag 10 PM. B also includes 11 PM, while A includes 9 PM instead. I'll isolate 11 PM to see whether that later hour helps explain B's advantage.

$$
A = \{21,22\}
$$

$$
9\text{ PM}, 10\text{ PM}
$$

$$
B = \{22,23\}
$$

$$
10\text{ PM}, 11\text{ PM}
$$

$$
\text{is\_concert\_window}=1
$$

$$
\text{is\_concert\_window}=0
$$

$$
B = 1
$$

$$
A = 0
$$

$$
\text{hour23\_only}
=
B \cap \neg A
$$


### Hour 23 Results - Testing Whether the Later Concert Window Captures Real Signal

The next comparison uses the same 11 PM rows for all alternatives.


In [ ]:
result_A_hour23 = run_comparison(
    weather_joined_sports_concerts_21_23, "A on hour 23", feature_cols, hour23_only_mask
)
result_B_hour23 = run_comparison(
    weather_joined_sports_concerts_22_24, "B on hour 23", feature_cols, hour23_only_mask
)
result_C_hour23 = run_comparison(
    weather_joined_sports, "C on hour 23", feature_cols_C, hour23_only_mask
)

pd.DataFrame([result_A_hour23, result_B_hour23, result_C_hour23])

B has much smaller errors on those rows: MAE 96.70 and RMSE 124.28. That supports including the later hour. It raises a follow-up question: does useful information continue into the first hour after midnight?

| Dataset | MAE | RMSE | $R^2$ |
|---|---:|---:|---:|
| A on hour 23 | 129.84 | 173.48 | -0.14 |
| B on hour 23 | **96.70** | **124.28** | **0.41** |
| C on hour 23 | 134.78 | 181.56 | -0.25 |


## Dataset D — Testing an Even Later Concert Window

I'll try D, covering 11 PM–1 AM. This introduces a date-boundary issue: midnight belongs to the day after the concert date. The expansion needs to carry the flag forward rather than mark the wrong midnight.

$$
11\text{ PM} \rightarrow 1\text{ AM}
$$

$$
\{23, 0\}
$$

$$
\text{August 10 at 11 PM}
\rightarrow
\text{August 11 at 12 AM}
$$

$$
\text{date} = \text{event\_date}
$$

$$
\text{hour}=23
$$

$$
\text{date}
=
\text{event\_date}+1\text{ day}
$$

$$
\text{hour}=0
$$

$$
\text{late concert window}
=
\text{same-day hour 23}
\lor
\text{next-day hour 0}
$$

$$
\text{June 5 at 00:00}
$$

$$
\text{June 5 at 23:00}
$$

$$
\text{June 6 at 00:00}
$$

$$
10\text{ PM} \rightarrow 12\text{ AM}
$$

$$
11\text{ PM} \rightarrow 1\text{ AM}
$$


In [ ]:
def add_late_concert_window_flag(df, concerts_df):
    """
    Flag zone-hour rows in an 11pm-1am window following a concert.
    Crosses midnight, so needs two conditions instead of one: hour 23
    on the concert's own date, OR hour 0 on the day AFTER the concert.
    """
    df = df.copy()
    concerts = concerts_df[["location_id", "event_date"]].drop_duplicates()
    df["date_only"] = df["hour"].dt.date

    # same-day part: hour 23 on the concert's own date
    same_day = concerts.copy()
    same_day["match_date"] = same_day["event_date"].dt.date
    same_day_merged = df.merge(
        same_day[["location_id", "match_date"]],
        left_on=["location_id", "date_only"],
        right_on=["location_id", "match_date"],
        how="left",
    )
    same_day_flag = same_day_merged["match_date"].notna() & (df["hour_of_day"] == 23)

    # next-day part: hour 0, checked against event_date + 1 day
    next_day = concerts.copy()
    next_day["match_date"] = (next_day["event_date"] + pd.Timedelta(days=1)).dt.date
    next_day_merged = df.merge(
        next_day[["location_id", "match_date"]],
        left_on=["location_id", "date_only"],
        right_on=["location_id", "match_date"],
        how="left",
    )
    next_day_flag = next_day_merged["match_date"].notna() & (df["hour_of_day"] == 0)

    df["is_late_concert_window"] = same_day_flag | next_day_flag
    df = df.drop(columns=["date_only"])
    return df

In [ ]:
weather_joined_sports_concerts_23_01 = add_late_concert_window_flag(
    weather_joined_sports, concerts_df
)

weather_joined_sports_concerts_23_01 = canonical_model_order(weather_joined_sports_concerts_23_01)

print(weather_joined_sports_concerts_23_01["is_late_concert_window"].sum())

In [ ]:
feature_cols_D = [c if c != "is_concert_window" else "is_late_concert_window" for c in feature_cols]

### Does the useful window continue after midnight?

B and D both include 11 PM, but only D includes the following midnight hour. I'll compare that hour directly to see what the extension adds.

$$
\{22,23\}
$$

$$
10\text{ PM} \rightarrow 12\text{ AM}
$$

$$
\{23,0\}
$$

$$
11\text{ PM} \rightarrow 1\text{ AM}
$$

$$
\boxed{\text{hour }0}
$$

$$
H:
\text{concert-window pickups remains detectable during hour 0}
$$


### Checking the midnight extension

I'll use the same midnight rows across the alternatives.


In [ ]:
hour0_only_mask = (
    weather_joined_sports_concerts_23_01["hour_of_day"] == 0
) & weather_joined_sports_concerts_23_01["is_late_concert_window"]

result_A_hour0 = run_comparison(
    weather_joined_sports_concerts_21_23, "A on hour 0", feature_cols, hour0_only_mask
)
result_B_hour0 = run_comparison(
    weather_joined_sports_concerts_22_24, "B on hour 0", feature_cols, hour0_only_mask
)
result_D_hour0 = run_comparison(
    weather_joined_sports_concerts_23_01, "D on hour 0", feature_cols_D, hour0_only_mask
)

pd.DataFrame([result_A_hour0, result_B_hour0, result_D_hour0])

D gives lower midnight errors, with MAE about 91.3 and RMSE 126.9. So the extension looks useful in this experiment. I'll now see whether keeping both the earlier core and the midnight extension gives a better combined window.

| Dataset | MAE | RMSE | $R^2$ |
|---|---:|---:|---:|
| A on hour 0 | 115.74 | 171.15 | -0.40 |
| B on hour 0 | 111.57 | 166.21 | -0.32 |
| D on hour 0 | **91.33** | **126.93** | **0.23** |


## Dataset E — Combining the Useful Hours Into a Wider 10 PM–1 AM Window

E combines hours 22, 23, and 0: 10 PM–1 AM. I'll compare it with B and D over that same three-hour period, so the score differences do not simply come from changing the evaluation hours.

$$
10\text{ PM} \rightarrow 1\text{ AM}
$$

$$
E = \{22,23,0\}
$$

$$
B = \{22,23\}
$$

$$
D = \{23,0\}
$$

$$
E = B \cup D
$$

$$
\text{is\_wide\_concert\_window}
=
\text{is\_concert\_window}
\lor
\text{is\_late\_concert\_window}
$$

$$
\{22,23,0\}
$$

$$
\{22,23,0\}
$$

$$
\boxed{
\text{10 PM--1 AM may capture the change in pickups around concerts better than either two-hour window alone}
}
$$


In [ ]:
same_keys = weather_joined_sports_concerts_22_24[["hour", "location_id"]].equals(
    weather_joined_sports_concerts_23_01[["hour", "location_id"]]
)

print("B and D keys aligned:", same_keys)

In [ ]:
# Dataset E: 10 PM - 1 AM concert-demand window

weather_joined_sports_concerts_22_01 = weather_joined_sports_concerts_22_24.copy()

weather_joined_sports_concerts_22_01["is_wide_concert_window"] = (
    weather_joined_sports_concerts_22_24["is_concert_window"].to_numpy()
    | weather_joined_sports_concerts_23_01["is_late_concert_window"].to_numpy()
)

weather_joined_sports_concerts_22_01 = canonical_model_order(weather_joined_sports_concerts_22_01)

print("E concert rows:", weather_joined_sports_concerts_22_01["is_wide_concert_window"].sum())

Each version has its own indicator name. I'll match the feature list to the dataset before fitting; otherwise a missing or wrong flag could make the comparison look like a timing effect.


### Comparing Dataset B, Dataset D, and Dataset E on the Same Three-Hour Window

The evaluation rows now cover the common 10 PM–1 AM period. Only the feature definition should differ.

$$
\{22,23,0\}
$$

$$
10\text{ PM} \rightarrow 1\text{ AM}
$$


In [ ]:
feature_cols_E = [c if c != "is_concert_window" else "is_wide_concert_window" for c in feature_cols]

# Evaluate all three models on the same 10 PM–1 AM rows
eval_mask_E = weather_joined_sports_concerts_22_01["is_wide_concert_window"]

result_B_hour_22_01 = run_comparison(
    weather_joined_sports_concerts_22_24,
    "B on 22pm - 01am",
    feature_cols=feature_cols,
    eval_mask=eval_mask_E,
)

result_D_hour_22_01 = run_comparison(
    weather_joined_sports_concerts_23_01,
    "D on 22pm - 01am",
    feature_cols=feature_cols_D,
    eval_mask=eval_mask_E,
)

result_E_hour_22_01 = run_comparison(
    weather_joined_sports_concerts_22_01,
    "E on 22pm - 01am",
    feature_cols=feature_cols_E,
    eval_mask=eval_mask_E,
)

pd.DataFrame([result_B_hour_22_01, result_D_hour_22_01, result_E_hour_22_01])

E has the lowest recorded MAE and RMSE, about 96.1 and 125.8. The next table keeps the three versions together for comparison.


The combined window performs better than either two-hour alternative on this shared period. I'll retain it as the working wide window, then check it against the no-concert control rather than just against other concert definitions.

| Dataset | Concert Window | MAE | RMSE | $R^2$ |
|---|---|---:|---:|---:|
| B | 10 PM–midnight | 100.50 | 138.16 | 0.32 |
| D | 11 PM–1 AM | 105.62 | 143.16 | 0.27 |
| E | 10 PM–1 AM | **96.11** | **125.78** | **0.44** |


## Final Concert-Feature Test — Dataset E vs. No-Concert Baseline

The question is now whether adding concert information helps at all. I'll compare C, without it, with the selected E window on the same event-related hours.

$$
10\text{ PM} \rightarrow 1\text{ AM}
$$

$$
\{22,23,0\}
$$

$$
\text{Does concert information improve prediction during concert-related hours?}
$$

$$
\text{Does one rare feature dramatically improve prediction across every NYC hour?}
$$

$$
C:
\text{no concert feature}
$$

$$
E:
\text{is\_wide\_concert\_window}
$$

$$
\text{weather + calendar + sports}
$$

$$
\text{weather + calendar + sports + concerts}
$$


### Final Concert-Feature Result — Dataset E vs. No-Concert Baseline

Both models will be scored on E's selected comparison rows.


In [ ]:
weather_joined_sports_C_model = canonical_model_order(weather_joined_sports)

eval_mask_final = weather_joined_sports_concerts_22_01["is_wide_concert_window"]

result_C_final = run_comparison(
    weather_joined_sports_C_model, "C - no concerts", feature_cols_C, eval_mask_final
)

result_E_final = run_comparison(
    weather_joined_sports_concerts_22_01, "E - 10pm-1am", feature_cols_E, eval_mask_final
)

pd.DataFrame([result_C_final, result_E_final])

Adding E lowers MAE from 126.5 to 96.1 and RMSE from 175.5 to 125.8. That is a substantial improvement within the explored period. Since these dates helped choose the window, I still need new-period evidence before treating the size of that gain as settled.

| Dataset | MAE | RMSE | $R^2$ |
|---|---:|---:|---:|
| C - no concerts | 126.51 | 175.49 | -0.10 |
| E - 10 PM–1 AM | **96.11** | **125.78** | **0.44** |


### Does the improvement hold outside concert hours?

That was the event-focused view. I'll also score the full 2026 period to see whether the local improvement changes overall performance or comes with costs elsewhere.

$$
\text{millions of ordinary zone-hours}
$$

$$
\text{a much smaller set of event-related zone-hours}
$$

$$
\text{full\_test\_mask}_i = True
$$

$$
C = \text{model without concert information}
$$

$$
E = \text{C model + the 10 PM--1 AM concert feature}
$$

$$
\text{large improvement during concerts}
$$

$$
\text{little or no change during the overall test period}
$$


### Whole Test Set Results — Does the Concert Feature Help Overall?

I'll keep this broader result separate from the event-period score.


In [ ]:
full_test_mask = pd.Series(True, index=weather_joined_sports.index)

result_C_whole = run_comparison(
    weather_joined_sports, "C - whole test set", feature_cols_C, full_test_mask
)
result_E_whole = run_comparison(
    weather_joined_sports_concerts_22_01, "E - whole test set", feature_cols_E, full_test_mask
)

pd.DataFrame([result_C_whole, result_E_whole])

The full-period result is almost unchanged, with E slightly worse under this initial zone representation. So the event benefit does not automatically become a pooled improvement. Before closing the comparison, I need to correct how the model receives zone IDs.

| Dataset | MAE | RMSE | R² |
|---|---:|---:|---:|
| C - whole test set | **41.56** | **65.80** | 0.70 |
| E - whole test set | 41.57 | 66.03 | 0.70 |


In [ ]:
variable_types = pd.DataFrame(
    {
        "dtype": weather_joined_sports_concerts_22_01[feature_cols_E].dtypes,
        "unique_values": weather_joined_sports_concerts_22_01[feature_cols_E].nunique(),
    }
)

variable_types

## A zone ID is a category, not a measurement

Zone IDs have been passed as numbers, but their order does not describe geography or activity. Zone 200 is not twice zone 100. I'll try categorical IDs so the model treats them as locations rather than an arbitrary numeric scale.

$$
\boxed{\text{location\_id is categorical, not continuous}}
$$

$$
\text{location\_id} \leq 120
$$

$$
\text{location\_id} > 120
$$

$$
\text{temperature} \leq 40^\circ F
$$

$$
186 \neq \text{continuous quantity}
$$

$$
186 = \text{category identifying one specific taxi zone}
$$


In [ ]:
weather_joined_sports_model = canonical_model_order(weather_joined_sports)

weather_joined_sports_concerts_model = canonical_model_order(weather_joined_sports_concerts_22_01)

weather_joined_sports_model["location_id"] = weather_joined_sports_model["location_id"].astype(
    "category"
)

weather_joined_sports_concerts_model["location_id"] = weather_joined_sports_concerts_model[
    "location_id"
].astype("category")

print("Engineering dtype:")
print(weather_joined_sports["location_id"].dtype)

print("\nCategorical C dtype:")
print(weather_joined_sports_model["location_id"].dtype)

print("\nCategorical E dtype:")
print(weather_joined_sports_concerts_model["location_id"].dtype)

### Testing the Effect of Treating Taxi Zones as Categories

I'll change the dtype in modeling copies and rerun C against E. Keeping the engineering table unchanged lets me isolate this representation choice from the earlier window choice.

$$
\text{location\_id: numeric code}
\rightarrow
\text{location\_id: categorical zone}
$$

$$
MAE_{\text{categorical zone}}
<
MAE_{\text{numeric zone}}
$$

$$
RMSE_{\text{categorical zone}}
<
RMSE_{\text{numeric zone}}
$$

$$
R^2_{\text{categorical zone}}
>
R^2_{\text{numeric zone}}
$$


### Results After Treating `location_id` as Categorical

Now I'll see how much that categorical treatment changes the comparison.


In [ ]:
full_test_mask_cat = pd.Series(True, index=weather_joined_sports_model.index)

result_C_whole_loc_cat = run_comparison(
    weather_joined_sports_model,
    "C - whole test set categorical location_id",
    feature_cols_C,
    full_test_mask_cat,
)

result_E_whole_loc_cat = run_comparison(
    weather_joined_sports_concerts_model,
    "E - whole test set categorical location_id",
    feature_cols_E,
    full_test_mask_cat,
)

pd.DataFrame([result_C_whole_loc_cat, result_E_whole_loc_cat])

Both models improve considerably. C has RMSE about 44.96 and E 44.86, giving E a small pooled advantage. The zone representation matters much more here than that small difference, while the event-period comparison explains why the concert feature remains worth keeping.

| Dataset | MAE | RMSE | $R^2$ |
|---|---:|---:|---:|
| C - categorical `location_id` | 25.08 | 44.96 | 0.86 |
| E - categorical `location_id` | **25.04** | **44.86** | **0.86** |


## What to keep from the concert experiments

I'll keep the approximate 10 PM–1 AM signal for later development. It gives useful predictive information in the explored event hours, but it is not a record of exact show endings. These choices also used dates that overlap later evaluation, so the final forecasting report needs to acknowledge that history.

$$
\boxed{\text{10 PM--1 AM}}
$$

$$
\text{is\_wide\_concert\_window}
=
1
$$

$$
\text{is\_wide\_concert\_window}
=
0
$$

$$
\boxed{\text{10 PM--1 AM concert pickup window}}
$$


## Saving the Engineered Datasets

I'll save the baseline and concert-enhanced tables so the next notebook can continue without repeating collection and joins. These are historical data products. The future-hour forecasting version will replace target-hour historical weather with appropriately selected forecast runs.


In [ ]:
gold_dir = PROJECT_ROOT / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Make export copies so the current notebook objects are not modified
baseline_export = (
    weather_joined_sports.sort_values(["hour", "location_id"]).reset_index(drop=True).copy()
)

concert_export = (
    weather_joined_sports_concerts_22_01.sort_values(["hour", "location_id"])
    .reset_index(drop=True)
    .copy()
)

# location_id is an identifier in the engineering layer.
# Categorical treatment will happen later in the modeling pipeline.
baseline_export["location_id"] = baseline_export["location_id"].astype("int64")
concert_export["location_id"] = concert_export["location_id"].astype("int64")

baseline_path = gold_dir / "zone_hour_pickups_weather_sports.parquet"
concert_path = gold_dir / "zone_hour_pickups_weather_sports_concerts.parquet"

baseline_export.to_parquet(baseline_path, index=False)

concert_export.to_parquet(concert_path, index=False)

print("Saved:")
print(baseline_path)
print(concert_path)

In [ ]:
baseline_check = pd.read_parquet(baseline_path)
concert_check = pd.read_parquet(concert_path)

print("=== BASELINE GOLD ===")
print("Shape:", baseline_check.shape)
print("location_id dtype:", baseline_check["location_id"].dtype)
print("Duplicate keys:", baseline_check.duplicated(subset=["hour", "location_id"]).sum())
print(
    "Sorted:",
    baseline_check[["hour", "location_id"]].equals(
        baseline_check[["hour", "location_id"]]
        .sort_values(["hour", "location_id"])
        .reset_index(drop=True)
    ),
)

print("\n=== CONCERT GOLD ===")
print("Shape:", concert_check.shape)
print("location_id dtype:", concert_check["location_id"].dtype)
print("Duplicate keys:", concert_check.duplicated(subset=["hour", "location_id"]).sum())
print("Concert rows:", concert_check["is_wide_concert_window"].sum())
print(
    "Sorted:",
    concert_check[["hour", "location_id"]].equals(
        concert_check[["hour", "location_id"]]
        .sort_values(["hour", "location_id"])
        .reset_index(drop=True)
    ),
)

## Saved historical pickup datasets

The two saved pickup tables have the same 12,668,184 rows. The baseline has 17 columns; adding concert fields brings the enhanced version to 19. I'll keep both for comparison, with filenames that distinguish their feature coverage rather than call them live production inputs.


In [ ]:
def is_control_day_clean(location_id, control_date, games_df, concerts_df):
    """
    True if a control day has neither a sports game nor a concert
    at the given zone — a genuinely clean comparison point.
    """
    has_game = (
        (games_df["location_id"] == location_id)
        & (games_df["start_time_et"].dt.date == control_date)
    ).any()
    has_concert = (
        (concerts_df["location_id"] == location_id)
        & (concerts_df["event_date"].dt.date == control_date)
    ).any()
    return not (has_game or has_concert)

## How long did riders who took a trip wait?

The trip records also let me prepare another outcome: the time from request to pickup. I'll use pickup rather than on-scene time because finding the rider can still take time after the driver arrives. This describes completed rides, not people whose requests never became trips.

$$
\text{wait time}
=
\text{pickup datetime}
-
\text{request datetime}
$$

$$
\text{pickup datetime}
<
\text{request datetime}
$$

I can calculate the time between request and pickup for a completed trip. Someone who leaves after seeing the estimated wait never enters this table, so the averages describe the riders whose trips went ahead. I use this table in the [pickup wait baseline](09_wait_time_baseline.ipynb).


In [ ]:
con.execute(f"""SELECT
                    COUNT(*) AS total_trips,
                    SUM(CASE WHEN pickup_datetime < request_datetime THEN 1 ELSE 0 END) AS negative_wait_counts,
                    AVG(EXTRACT(EPOCH FROM(pickup_datetime - request_datetime))) AS avg_wait_seconds
                FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'""").df()

### How can a pickup come before a request?

There are about 13 million negative durations. Those cannot represent real elapsed waits, so I'll exclude them before calculating the outcome. The earlier average of 294.85 seconds includes them and should not be reported as the cleaned estimate.

$$
\text{pickup datetime}
<
\text{request datetime}
$$

$$
294.85 \text{ seconds}
$$

$$
\text{pickup datetime} \geq \text{request datetime}
$$


### Could anyone really have waited that long?

Removing negatives is only the first step. I'll inspect the largest remaining durations before deciding where to set a practical upper limit.


In [ ]:
con.execute(f"""SELECT
                    MIN(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime))) AS min_wait_seconds,
                    MAX(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime))) AS max_wait_seconds,
                    AVG(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime))) AS avg_wait_seconds_clean,
            COUNT(*) AS clean_trip_count
            FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
            WHERE pickup_datetime >= request_datetime""").df()

The mean rises to about 303 seconds, but the maximum is roughly 31 days. That is not a credible ride wait. I'll look farther into the distribution rather than choose a cutoff from the maximum alone.

$$
2{,}676{,}835 \text{ seconds}
$$


In [ ]:
con.execute(f"""SELECT
                    approx_quantile(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)), 0.50) AS p50,
                    approx_quantile(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)), 0.95) AS p95,
                    approx_quantile(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)), 0.99) AS p99,
                    approx_quantile(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)), 0.999) AS p999
                FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
                WHERE pickup_datetime >= request_datetime""").df()

### Where should I cap the recorded waits?

Most waits are much shorter: the median is about 4.3 minutes and the 99.9th percentile 28.7 minutes. I'll inspect the farther tail before selecting a rule that removes obvious timestamp problems without discarding many long but plausible waits.


In [ ]:
con.execute(f"""SELECT
                    approx_quantile(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)), 0.9999) AS p9999,
                    SUM(CASE WHEN EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)) > 3600 THEN 1 ELSE 0 END) AS over_1hr,
                    SUM(CASE WHEN EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)) > 7200 THEN 1 ELSE 0 END) AS over_2hr,
                    COUNT(*) AS total
                FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
                WHERE pickup_datetime >= request_datetime
                """).df()

### Final Wait-Time Cleaning Rule

I'll keep durations from zero to one hour. The 99.99th percentile is about 51.8 minutes, and only 59,724 nonnegative records exceed an hour. This is a practical cleaning choice, not proof that every longer wait is impossible.

$$
3108 \text{ seconds}
$$

$$
51.8 \text{ minutes}
$$

$$
0.005\%
$$

$$
0
\leq
\text{wait time}
\leq
3600 \text{ seconds}
$$


In [ ]:
wait_time_hourly = con.execute(f"""
    SELECT
        date_trunc('hour', pickup_datetime) AS hour,
        PULocationID AS location_id,
        AVG(EXTRACT(EPOCH FROM (pickup_datetime - request_datetime))) AS avg_wait_seconds,
        COUNT(*) AS n_trips_with_wait_data
    FROM '{PROJECT_ROOT}/data/bronze/fhvhv_tripdata_*.parquet'
    WHERE pickup_datetime >= request_datetime
      AND EXTRACT(EPOCH FROM (pickup_datetime - request_datetime)) <= 3600
      AND PULocationID BETWEEN 1 AND 263
    GROUP BY
        date_trunc('hour', pickup_datetime),
        PULocationID
""").df()

In [ ]:
wait_time_hourly.shape

### Validating the Hourly Wait-Time Table

The hourly waiting-time table has fewer rows than the complete pickup grid, which is expected. An average needs at least one retained trip. I'll check the missing combinations against trip counts and exclusions, then leave an undefined wait as missing rather than call it a zero-minute wait.

$$
12{,}668{,}184
$$

$$
12{,}095{,}075
$$


In [ ]:
# Complete demand grid
total_grid_rows = len(weather_joined_sports)

# Final cleaned wait-time table
wait_rows = len(wait_time_hourly)

# Demand hours with no pickups
zero_pickup_hours = (weather_joined_sports["total_pickups"] == 0).sum()

# Hours with pickups but no valid wait-time observations
has_pickups = weather_joined_sports.loc[
    weather_joined_sports["total_pickups"] > 0, ["location_id", "hour", "total_pickups"]
]

missing_wait = has_pickups.merge(
    wait_time_hourly[["location_id", "hour"]],
    on=["location_id", "hour"],
    how="left",
    indicator=True,
)

missing_wait = missing_wait[missing_wait["_merge"] == "left_only"]

# Check that wait-time keys stay inside the canonical grid
grid_keys = weather_joined_sports[["location_id", "hour"]].drop_duplicates()

wait_outside_grid = wait_time_hourly[["location_id", "hour"]].merge(
    grid_keys, on=["location_id", "hour"], how="left", indicator=True
)

wait_outside_grid = wait_outside_grid[wait_outside_grid["_merge"] == "left_only"]

print(f"Complete grid rows: {total_grid_rows:,}")
print(f"Wait-time rows: {wait_rows:,}")
print(f"Zone-hours without wait rows: {total_grid_rows - wait_rows:,}")
print(f"Zero-pickup zone-hours: {zero_pickup_hours:,}")
print("Pickup zone-hours with no valid wait data:", f"{len(missing_wait):,}")
print("Wait-time keys outside canonical grid:", f"{len(wait_outside_grid):,}")

print("\nPickup counts where wait data is missing:")
print(missing_wait["total_pickups"].describe())

## Saving the Hourly Wait-Time Dataset

I'll save waiting time as a separate outcome table. If I used the realized average for a future hour to predict pickups in that hour, I would be giving the forecast information from trips that had not happened yet. Separating the products helps avoid that mistake.

$$
0
\leq
\text{wait time}
\leq
3600 \text{ seconds}
$$


In [ ]:
import pandas as pd

In [ ]:
wait_time_path = PROJECT_ROOT / "data" / "gold" / "zone_hour_wait_time.parquet"

wait_time_hourly.to_parquet(wait_time_path, index=False)

print("Saved:")
print(wait_time_path)

In [ ]:
wait_time_check = pd.read_parquet(PROJECT_ROOT / "data" / "gold" / "zone_hour_wait_time.parquet")

print("Shape:", wait_time_check.shape)

print("\nDtypes:")
print(wait_time_check.dtypes)

print("\nMissing values:")
print(wait_time_check.isna().sum())

print("\nDuplicate location_id-hour keys:")
print(wait_time_check.duplicated(subset=["location_id", "hour"]).sum())

print("\nWait-time range:")
print(wait_time_check["avg_wait_seconds"].agg(["min", "max"]))

print("\nTrip-count range:")
print(wait_time_check["n_trips_with_wait_data"].agg(["min", "max"]))

## How much can earlier pickup counts tell me?

An earlier model barely moved above its training average even as later pickups increased. Calendar and location alone were not enough to track the changing level. I'll add counts from 24, 168, and 336 hours before each target to give the model more recent and recurring history.

$$
\text{pickup\_lag\_24}_{z,t}=y_{z,t-24}
$$

$$
\text{pickup\_lag\_168}_{z,t}=y_{z,t-168}
$$

$$
\text{pickup\_lag\_336}_{z,t}=y_{z,t-336}
$$


In [ ]:
lag_source_data = pd.read_parquet(
    PROJECT_ROOT/'data'/'gold'/'zone_hour_pickups_weather_E_with_concerts.parquet',
    columns=[
        "hour",
        "location_id",
        "total_pickups"
    ]
)

In [ ]:
lag_source_data = (
    lag_source_data
    .sort_values(
        ["location_id", "hour"]
    )
    .reset_index(drop=True)
)

In [ ]:
print("Shape:", lag_source_data.shape)
print("First hour:", lag_source_data["hour"].min())
print("Last hour:", lag_source_data["hour"].max())
print("Locations:", lag_source_data["location_id"].nunique())

duplicate_zone_hours = (
    lag_source_data
    .duplicated(["location_id", "hour"])
    .sum()
)

print("Duplicate zone-hours:", duplicate_zone_hours)

In [ ]:
lag_source_data["pickup_lag_24"] = (
    lag_source_data
    .groupby(
        "location_id",
        sort=False
    )["total_pickups"]
    .shift(24)
    .astype("float32")
)

In [ ]:
missing_lag_24 = (
    lag_source_data["pickup_lag_24"]
    .isna()
    .sum()
)

expected_missing_lag_24 = 263 * 24

print("Missing lag-24 values:", missing_lag_24)
print("Expected missing values:", expected_missing_lag_24)
print(
    "Validation passed:",
    missing_lag_24 == expected_missing_lag_24
)

In [ ]:
lag_source_data.loc[
    lag_source_data["location_id"].eq(3),
    [
        "hour",
        "location_id",
        "total_pickups",
        "pickup_lag_24"
    ]
].head(28)

In [ ]:
print("Missing lag-24 values:", missing_lag_24)
print("Expected missing values:", 263 * 24)
print("Validation passed:", missing_lag_24 == 263 * 24)

In [ ]:
lag_source_data["pickup_lag_168"] = (
    lag_source_data
    .groupby(
        "location_id",
        sort=False
    )["total_pickups"]
    .shift(168)
    .astype("float32")
)

missing_lag_168 = (
    lag_source_data["pickup_lag_168"]
    .isna()
    .sum()
)

expected_missing_lag_168 = 263 * 168

print("Missing lag-168 values:", missing_lag_168)
print("Expected missing values:", expected_missing_lag_168)
print(
    "Validation passed:",
    missing_lag_168 == expected_missing_lag_168
)

In [ ]:
lag_source_data.loc[
    (
        lag_source_data["location_id"].eq(3)
        & lag_source_data["hour"].between(
            "2021-01-01 00:00:00",
            "2021-01-08 03:00:00"
        )
    ),
    [
        "hour",
        "location_id",
        "total_pickups",
        "pickup_lag_168"
    ]
].head(5)

In [ ]:
lag_source_data["pickup_lag_336"] = (
    lag_source_data
    .groupby(
        "location_id",
        sort=False
    )["total_pickups"]
    .shift(336)
    .astype("float32")
)

In [ ]:
missing_lag_336 = (
    lag_source_data["pickup_lag_336"]
    .isna()
    .sum()
)

expected_missing_lag_336 = 263 * 336

print("Missing lag-336 values:", missing_lag_336)
print("Expected missing values:", expected_missing_lag_336)
print(
    "Validation passed:",
    missing_lag_336 == expected_missing_lag_336
)

In [ ]:
lag_columns = [
    "pickup_lag_24",
    "pickup_lag_168",
    "pickup_lag_336"
]

lag_availability = pd.DataFrame({
    "training": (
        lag_source_data.loc[
            lag_source_data["hour"] < "2025-01-01",
            lag_columns
        ]
        .isna()
        .sum()
    ),
    "validation": (
        lag_source_data.loc[
            lag_source_data["hour"].between(
                "2025-01-01",
                "2025-12-31 23:00:00"
            ),
            lag_columns
        ]
        .isna()
        .sum()
    ),
    "test": (
        lag_source_data.loc[
            lag_source_data["hour"] >= "2026-01-01",
            lag_columns
        ]
        .isna()
        .sum()
    )
})

lag_availability

In [ ]:
pickup_lag_features = lag_source_data[
    [
        "hour",
        "location_id",
        "pickup_lag_24",
        "pickup_lag_168",
        "pickup_lag_336"
    ]
].copy()

In [ ]:
PICKUP_LAG_PATH = (
     PROJECT_ROOT/'data'/'gold'
    / "zone_hour_pickup_lags.parquet"
)

pickup_lag_features.to_parquet(
    PICKUP_LAG_PATH,
    index=False,
    engine="pyarrow",
    compression="zstd"
)

In [ ]:
print("File exists:", PICKUP_LAG_PATH.exists())
print(
    "File size (MB):",
    round(
        PICKUP_LAG_PATH.stat().st_size / 1024**2,
        2
    )
)
print("Rows:", len(pickup_lag_features))
print("Columns:", pickup_lag_features.shape[1])
print(pickup_lag_features.dtypes)

I calculate the lags before splitting so the start of validation can use legitimate training-period history. Missing values should occur at the beginning, where no earlier count exists. The later forecast pipeline will still check source timing and the availability of the cleaned counts, including the daylight-saving issue discussed above.

| Feature | Training Missing | Validation Missing | Test Missing |
|:--|--:|--:|--:|
| `pickup_lag_24` | 6,312 | 0 | 0 |
| `pickup_lag_168` | 44,184 | 0 | 0 |
| `pickup_lag_336` | 88,368 | 0 | 0 |

$$
263\times336=88,368
$$

$$
T+h-24\le T
$$


## What the next stage can build on

My first count missed the hours with zero pickups. Fixing that gave me a table I could check after every join. I could then add calendar and event features and keep the wait-time table separate.

The next question is which inputs I would actually have when making a forecast. I use the [archived weather notebook](03_archived_weather_forecast_production_updated.ipynb) to check which forecasts were available at each prediction time. Then I look at the finished table in the [forecasting data exploration](03b_production_dataset_eda.ipynb).

$$
10\text{ PM} \text{ through } 1\text{ AM}
$$

$$
\text{wait time}
=
\text{pickup datetime}
-
\text{request datetime}
$$

$$
0
\leq
\text{wait time}
\leq
3600 \text{ seconds}
$$


In [ ]:
print("=== MODEL INPUT CHECK ===")

print("\nBaseline shape:")
print(weather_joined_sports.shape)

print("\nBaseline target sum:")
print(weather_joined_sports["total_pickups"].sum())

print("\nDataset A shape:")
print(weather_joined_sports_concerts_21_23.shape)

print("\nDataset A concert rows:")
print(weather_joined_sports_concerts_21_23["is_concert_window"].sum())

print("\nDataset B shape:")
print(weather_joined_sports_concerts_22_24.shape)

print("\nDataset B concert rows:")
print(weather_joined_sports_concerts_22_24["is_concert_window"].sum())

print("\nHour range:")
print(weather_joined_sports["hour"].min())
print(weather_joined_sports["hour"].max())

print("\nDuplicate baseline keys:")
print(weather_joined_sports.duplicated(subset=["location_id", "hour"]).sum())

print("\nFeature columns:")
print(feature_cols)

In [ ]:
import hashlib


def dataframe_fingerprint(df, columns):
    row_hashes = pd.util.hash_pandas_object(df[columns], index=False).to_numpy()

    # Sensitive to both values AND current row order
    ordered_hash = hashlib.sha256(row_hashes.tobytes()).hexdigest()

    # Sensitive to values, but NOT row order
    sorted_hashes = np.sort(row_hashes.copy())

    content_hash = hashlib.sha256(sorted_hashes.tobytes()).hexdigest()

    return ordered_hash, content_hash


baseline_check_cols = ["hour"] + feature_cols_C + ["total_pickups"]

ordered_hash, content_hash = dataframe_fingerprint(weather_joined_sports, baseline_check_cols)

print("Baseline ordered fingerprint:")
print(ordered_hash)

print("\nBaseline content fingerprint:")
print(content_hash)

In [ ]:
result_A = run_comparison(
    weather_joined_sports_concerts_21_23, "A (9-11pm)", feature_cols, eval_mask
)

result_B = run_comparison(
    weather_joined_sports_concerts_22_24, "B (10pm-midnight)", feature_cols, eval_mask
)

result_C = run_comparison(weather_joined_sports, "C (no concerts)", feature_cols_C, eval_mask)

pd.DataFrame([result_A, result_B, result_C])

In [ ]:
sort_cols = ["hour", "location_id"]

df_A_model = weather_joined_sports_concerts_21_23.sort_values(sort_cols).reset_index(drop=True)

df_B_model = weather_joined_sports_concerts_22_24.sort_values(sort_cols).reset_index(drop=True)

df_C_model = weather_joined_sports.sort_values(sort_cols).reset_index(drop=True)

print(df_A_model[["hour", "location_id"]].head())
print(df_A_model[["hour", "location_id"]].tail())

In [ ]:
eval_mask_model = df_A_model["is_concert_window"] | df_B_model["is_concert_window"]

result_A = run_comparison(df_A_model, "A (9-11pm)", feature_cols, eval_mask_model)

result_B = run_comparison(df_B_model, "B (10pm-midnight)", feature_cols, eval_mask_model)

result_C = run_comparison(df_C_model, "C (no concerts)", feature_cols_C, eval_mask_model)

pd.DataFrame([result_A, result_B, result_C])